# 1. Research Objective and Experiment Overview
Static-K AutoDSPy GRPO on official GSM8K `main`. Independent GPT-2 policies use K=5,8,10,15, Llama 3.1 8B through Ollama, and the **Figure 2 sum loss**. This is a reduced-budget controlled experiment, not reproduction of published scores.

One episode samples **one** question with replacement from a fixed training pool, attempts K pipelines, and makes at most one Adam step. A 200-question pool and 200 episodes need not encounter 200 unique questions. Figure 2 and the released loop support this protocol; appendix wording about 200 samples per episode conflicts. See [audit](docs/PHASE_1_AUDIT.md) and [master specification](docs/MASTER_SPEC.md).

Run definition/setup cells top to bottom in a fresh kernel. Operations are explicitly invoked in section 21. Setup does not train or test. Everything executable is in this notebook; HotpotQA has its own independent notebook.

# 2. Environment and Dependency Setup
Use the existing GPU virtual environment as your Jupyter kernel. Installation is opt-in and does not replace your PyTorch build. Install CUDA-compatible PyTorch separately if needed. Version snapshots accompany every run; frozen versions alone do not guarantee Ollama determinism.

In [1]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "dspy==3.3.0", "transformers==5.14.1",
                           "datasets==5.0.1", "numpy>=2.0,<3", "matplotlib>=3.10,<4", "nbformat>=5.10", "ipykernel"])

import copy, gc, hashlib, importlib.metadata, json, math, os, random, re, sys, tempfile, time, uuid
from contextlib import contextmanager
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from pathlib import Path
from urllib.request import Request, urlopen
import numpy as np
import torch
import dspy
import matplotlib.pyplot as plt
from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import GPT2LMHeadModel, GPT2Tokenizer

VERSIONS = {name: importlib.metadata.version(name) for name in
            ["torch", "transformers", "dspy", "datasets", "numpy", "matplotlib"]}
print("Python:", sys.version, "\nPackages:", VERSIONS, "\nCUDA:", torch.cuda.is_available())

x:\Adaptive GRPO for AutoDSPy\Code\AUTODSPy\venv_gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)] 
Packages: {'torch': '2.6.0+cu124', 'transformers': '5.14.1', 'dspy': '3.3.0', 'datasets': '5.0.1', 'numpy': '2.4.4', 'matplotlib': '3.11.1'} 
CUDA: True


# 3. Experiment Configuration
Edit these controls, then call `build_config()` again. Smoke/pilot/baseline default to 5/20/200 episodes using the same 200-example pool and persisted 200-episode schedule. Smaller explicit sizes are engineering checks. GPT-2 (paper label 127M), Adam 1e-4, and baseline K=5 follow reported settings; K=8, seed, subsets and resource/decoding settings are project choices.

`OLLAMA_NUM_GPU=0` offloads Llama to CPU, useful when GPT-2 and its optimizer share a 6GB GPU. Change it explicitly for partial/automatic offload; keep it identical across compared K runs. No model substitution occurs.

In [ ]:
K = 5
SEED = 42
PRESET = "smoke"                       # smoke / pilot / baseline
MODE = "initialize"                    # initialize / train / resume / validate / test / compare
NUM_EPISODES = None                     # None chooses preset; override freely
TRAIN_POOL_SIZE = 200
VALIDATION_SIZE = 50
TEST_SIZE = 200
OLLAMA_MODEL = "llama3.1:8b"
OLLAMA_HOST = "http://localhost:11434"
CHECKPOINT_INTERVAL = 25
VALIDATION_INTERVAL = None              # baseline 50; smoke 5; pilot 10
SCHEDULE_EPISODES = 200                 # common maximum; presets use prefixes
POLICY_MODEL = "gpt2"
POLICY_REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"
DATASET_REVISION = None                 # resolve main to immutable HF commit before loading
LEARNING_RATE = 1e-4
POLICY_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OLLAMA_NUM_GPU = 0                      # 0=CPU; None=Ollama default; integer=GPU layers
OLLAMA_NUM_CTX = 4096
EXECUTION_MAX_TOKENS = 512
EXECUTION_TEMPERATURE = 0.0
REQUEST_TIMEOUT = 180
RESULTS_ROOT = Path("results")
SELECTED_CHECKPOINT = None              # explicit path for resume / validate / test
RUN_OPERATION = False                  # section 21 safety switch
RUN_ALL_K = False                      # explicit sequential fresh starts; never auto-enabled
ALLOWED_K = (5, 8, 10, 15)
PROTOCOL_VERSION = "gsm8k-static-figure2-v1"

def build_config(**overrides):
    presets = {"smoke": 5, "pilot": 20, "baseline": 200}
    if PRESET not in presets:
        raise ValueError("PRESET must be smoke, pilot or baseline")
    cfg = dict(dataset="gsm8k", dataset_name="openai/gsm8k", dataset_config="main",
               dataset_revision=DATASET_REVISION, policy_model=POLICY_MODEL, policy_revision=POLICY_REVISION,
               k=K, seed=SEED, preset=PRESET, num_episodes=NUM_EPISODES if NUM_EPISODES is not None else presets[PRESET],
               train_pool_size=TRAIN_POOL_SIZE, validation_size=VALIDATION_SIZE, test_size=TEST_SIZE,
               schedule_episodes=SCHEDULE_EPISODES, learning_rate=LEARNING_RATE,
               checkpoint_interval=CHECKPOINT_INTERVAL,
               validation_interval=VALIDATION_INTERVAL if VALIDATION_INTERVAL is not None else {"smoke":5,"pilot":10,"baseline":50}[PRESET],
               policy_device=POLICY_DEVICE, ollama_model=OLLAMA_MODEL, ollama_host=OLLAMA_HOST.rstrip("/"),
               ollama_num_gpu=OLLAMA_NUM_GPU, ollama_num_ctx=OLLAMA_NUM_CTX,
               execution_max_tokens=EXECUTION_MAX_TOKENS, execution_temperature=EXECUTION_TEMPERATURE,
               request_timeout=REQUEST_TIMEOUT, results_root=str(RESULTS_ROOT), protocol=PROTOCOL_VERSION,
               policy_dropout=False, policy_sampling="restricted-first-token-multinomial",
               loss="negative-sum-logprob-times-centered-reward", optimizer="Adam", cache=False,
               semantic_fallback=True, max_policy_tokens=1024, max_pipeline_steps=3,
               evaluation_sampling="multinomial-with-fixed-split-seed", evaluation_seed_offset=100000)
    cfg.update(overrides)
    if cfg["k"] not in ALLOWED_K:
        raise ValueError(f"K must be one of {ALLOWED_K}")
    for key in ["num_episodes","train_pool_size","validation_size","test_size","schedule_episodes",
                "checkpoint_interval","validation_interval","execution_max_tokens","ollama_num_ctx"]:
        if not isinstance(cfg[key], int) or cfg[key] <= 0:
            raise ValueError(f"{key} must be a positive integer")
    cfg["schedule_episodes"] = max(cfg["schedule_episodes"], cfg["num_episodes"])
    if cfg["learning_rate"] <= 0 or cfg["policy_device"] not in ("cpu", "cuda"):
        raise ValueError("Use positive learning rate and cpu/cuda policy_device")
    return cfg

CONFIG = build_config()
print(CONFIG)

# 4. Hardware and Ollama Initialization
No server is launched or model pulled automatically. Start Ollama yourself and run `ollama pull llama3.1:8b` if absent. The endpoint is configurable, including a remote machine.

The instrumented original `dspy.LM('ollama_chat/...')` counts LM **forward attempts**, including adapter retries and semantic judges; these are logical calls. Exact HTTP request counts are unavailable because provider internals are not instrumented. Cache is disabled and provider retries are zero. Returned token usage is measured when present; failures may have unreported tokens. Ollama `/api/ps` reports server model residency, distinct from PyTorch allocator memory.

In [ ]:
def json_request(host, route, payload=None, timeout=15):
    data = None if payload is None else json.dumps(payload).encode()
    request = Request(host.rstrip("/")+route, data=data, headers={"Content-Type":"application/json"})
    with urlopen(request, timeout=timeout) as response:
        return json.load(response)

def ollama_snapshot(cfg):
    try:
        return json_request(cfg["ollama_host"], "/api/ps", timeout=5)
    except Exception as exc:
        return {"unavailable": f"{type(exc).__name__}: {exc}"}

def judge_inference_parameters(lm, overrides=None):
    # Record provider request settings, never authentication keys or headers.
    configured = dict(getattr(lm, "kwargs", {}) or {})
    configured.update(overrides or {})
    allowed = ("api_base", "temperature", "max_tokens", "seed", "num_ctx", "num_gpu",
               "top_p", "top_k", "repeat_penalty", "stop", "timeout", "keep_alive", "rollout_id")
    return dict(model=getattr(lm,"model",None), model_type=getattr(lm,"model_type",None),
                cache=configured.get("cache",getattr(lm,"cache",None)),
                num_retries=getattr(lm,"num_retries",None),
                parameters={key:copy.deepcopy(configured[key]) for key in allowed if key in configured},
                scope="DSPy/LiteLLM request settings; physical HTTP/server settings not observed")

class MeasuredLM(dspy.LM):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.events = []
        self.capture_semantic_judge = False
    def forward(self, *args, **kwargs):
        event = {"seconds": None, "prompt_tokens": None, "generated_tokens": None, "error": None}
        if self.capture_semantic_judge:
            event["judge_request"] = dict(prompt=kwargs.get("prompt",args[0] if args else None),
                                          messages=copy.deepcopy(kwargs.get("messages")),
                                          inference=judge_inference_parameters(self,kwargs))
        started = time.perf_counter()
        try:
            response = super().forward(*args, **kwargs)
            if self.capture_semantic_judge:
                event["judge_provider_response"] = (response.model_dump(mode="json")
                    if hasattr(response,"model_dump") else str(response))
            usage = dict(getattr(response, "usage", {}) or {})
            event["prompt_tokens"] = usage.get("prompt_tokens")
            event["generated_tokens"] = usage.get("completion_tokens")
            return response
        except Exception as exc:
            event["error"] = f"{type(exc).__name__}: {exc}"
            raise
        finally:
            event["seconds"] = time.perf_counter()-started
            self.events.append(event)

def initialize_ollama(cfg):
    try:
        version = json_request(cfg["ollama_host"], "/api/version")
        tags = json_request(cfg["ollama_host"], "/api/tags")["models"]
        model = next((m for m in tags if m.get("name")==cfg["ollama_model"] or m.get("model")==cfg["ollama_model"]), None)
        if model is None:
            raise RuntimeError(f"Missing {cfg['ollama_model']}; run ollama pull {cfg['ollama_model']}")
        show = json_request(cfg["ollama_host"], "/api/show", {"model":cfg["ollama_model"]})
    except Exception as exc:
        raise RuntimeError(f"Ollama unavailable at {cfg['ollama_host']}: {exc}") from exc
    options = dict(api_base=cfg["ollama_host"], api_key="", cache=False, num_retries=0,
                   temperature=cfg["execution_temperature"], max_tokens=cfg["execution_max_tokens"],
                   seed=cfg["seed"], num_ctx=cfg["ollama_num_ctx"], timeout=cfg["request_timeout"], keep_alive="5m")
    if cfg["ollama_num_gpu"] is not None:
        options["num_gpu"] = cfg["ollama_num_gpu"]
    lm = MeasuredLM("ollama_chat/"+cfg["ollama_model"], **options)
    dspy.configure(lm=lm)
    metadata = {"version":version, "model":model, "details":show.get("details"),
                "model_info":show.get("model_info"), "options":options, "ps":ollama_snapshot(cfg)}
    return lm, metadata

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)

def restore_rng(states):
    random.setstate(states["python"]); np.random.set_state(states["numpy"]); torch.set_rng_state(states["torch"])
    if states["cuda"] is not None:
        if not torch.cuda.is_available():
            raise RuntimeError("Checkpoint has CUDA RNG state; resume in its original GPU environment")
        torch.cuda.set_rng_state_all(states["cuda"])

@contextmanager
def isolated_rng(seed):
    saved = rng_state()
    seed_everything(seed)
    try:
        yield
    finally:
        restore_rng(saved)

def hardware_metadata():
    return dict(python=sys.version, packages=VERSIONS, cuda_available=torch.cuda.is_available(),
                cuda_version=torch.version.cuda,
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                gpu_total_bytes=torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None)

# 5. Dataset Loading
Official [GSM8K](https://huggingface.co/datasets/openai/gsm8k), `main`. Resolve an immutable dataset revision before downloading. Loading preserves complete train/test splits; sampling happens separately. Ground truth is the numeric answer after `####`, while the rationale stays in dataset metadata and is never passed to the policy or execution module.

In [ ]:
def canonical_number(text):
    try:
        number = Decimal(str(text).strip().replace(",", ""))
        if not number.is_finite():
            return None
        return "0" if number == 0 else format(number.normalize(), "f")
    except (InvalidOperation, ValueError):
        return None

def canonical_target(answer):
    if "####" not in answer:
        raise ValueError("GSM8K answer lacks #### delimiter")
    target = canonical_number(answer.rsplit("####",1)[1])
    if target is None:
        raise ValueError(f"Invalid GSM8K numeric target: {answer!r}")
    return target

def load_gsm8k(cfg):
    cfg = copy.deepcopy(cfg)
    try:
        if cfg["dataset_revision"] is None:
            cfg["dataset_revision"] = HfApi().dataset_info(cfg["dataset_name"]).sha
        datasets = load_dataset(cfg["dataset_name"], cfg["dataset_config"], revision=cfg["dataset_revision"])
    except Exception as exc:
        raise RuntimeError("Cannot load official GSM8K. Check network/cache and dataset revision: "+str(exc)) from exc
    return cfg, datasets

def dataset_example(dataset, split, index):
    item = dataset[split][int(index)]
    return dict(id=f"gsm8k:{split}:{index}", split=split, index=int(index), question=item["question"],
                answer=canonical_target(item["answer"]), rationale=item["answer"])

def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path, value, overwrite=False):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite:
        raise FileExistsError(f"Refusing overwrite: {path}")
    tmp = path.with_name(path.name+"."+uuid.uuid4().hex+".tmp")
    with tmp.open("w",encoding="utf-8") as stream:
        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
        stream.flush(); os.fsync(stream.fileno())
    if overwrite:
        os.replace(tmp,path)
    else:
        os.link(tmp,path)                  # exclusive publication; concurrent writer cannot overwrite
        tmp.unlink()

# 6. Dataset Sampling and Episode Schedule
A dataset/seed/size/revision manifest is shared across K and presets. Seeded permutation separates training and validation. A separate seeded generator samples training-pool offsets **with replacement** once, then persists the full schedule. Validation/test indices and content hashes are fixed. Changing sizes/revision creates a different protocol namespace; never compare those as controlled K runs.

Independent deterministic streams use base seed for train permutation, seed+1 for test permutation, and seed+2 for the replacement schedule. These are project choices derived from SEED=42; no K-specific RNG generates the question schedule.

In [ ]:
def selection_payload(cfg, datasets):
    ntrain, ntest = len(datasets["train"]), len(datasets["test"])
    if cfg["train_pool_size"]+cfg["validation_size"] > ntrain or cfg["test_size"] > ntest:
        raise ValueError("Requested subset exceeds official split; reduce configured sizes")
    permutation = np.random.default_rng(cfg["seed"]).permutation(ntrain)
    train_idx = permutation[:cfg["train_pool_size"]].tolist()
    val_idx = permutation[cfg["train_pool_size"]:cfg["train_pool_size"]+cfg["validation_size"]].tolist()
    test_idx = np.random.default_rng(cfg["seed"]+1).permutation(ntest)[:cfg["test_size"]].tolist()
    subsets = {"train":[dataset_example(datasets,"train",i) for i in train_idx],
               "validation":[dataset_example(datasets,"train",i) for i in val_idx],
               "test":[dataset_example(datasets,"test",i) for i in test_idx]}
    schedule = np.random.default_rng(cfg["seed"]+2).integers(0,cfg["train_pool_size"],size=cfg["schedule_episodes"]).tolist()
    identity = {key:cfg[key] for key in ["dataset_name","dataset_config","dataset_revision","seed",
                                        "train_pool_size","validation_size","test_size","schedule_episodes"]}
    return dict(identity=identity, indices=dict(train=train_idx,validation=val_idx,test=test_idx),
                ids={name:[item["id"] for item in rows] for name,rows in subsets.items()},
                content_hashes={name:stable_hash(rows) for name,rows in subsets.items()},
                episode_pool_offsets=schedule,
                episode_question_ids=[subsets["train"][offset]["id"] for offset in schedule]), subsets

def prepare_selection(cfg, datasets):
    payload, subsets = selection_payload(cfg,datasets)
    selection_id = stable_hash(payload)
    path = Path(cfg["results_root"])/"gsm8k"/"selections"/(stable_hash(payload["identity"])+".json")
    if path.exists():
        persisted = json.loads(path.read_text(encoding="utf-8"))
        if persisted != payload:
            raise RuntimeError("Persisted dataset selection/schedule mismatch; do not overwrite it")
    else:
        write_json(path,payload)
    assert set(payload["indices"]["train"]).isdisjoint(payload["indices"]["validation"])
    assert set(payload["ids"]["train"]).isdisjoint(payload["ids"]["test"])
    return dict(subsets=subsets, selection=payload, selection_id=selection_id, selection_path=str(path),
                full_test=datasets["test"])

# Repeated draws are allowed, not forced. Small schedules may happen to contain no duplicate.
# Schedule identity never depends on K or preset; episodes use its leading prefix.

# 7. DSPy Action Space and Signatures
The released module pool and all fifteen signatures are retained, including signatures that do not naturally fit math QA. The policy selects one module, one signature, then deterministic `stop`; this released toy implementation therefore executes one DSPy stage per pipeline. No ReAct, retrieval, teleprompter, or new action head.

In [ ]:
MODULES = ["CoT", "Predict"]
SIGNATURES = ["question -> answer", "text -> summary", "question -> reasoning", "question -> hypothesis",
              "problem -> solution", "problem_description -> explanation", "context -> summary", "context -> briefing",
              "word_problem -> solution", "math_problem -> answer", "prompt -> response", "query -> response",
              "text -> response", "prompt -> generated_text", "query -> generated_text"]
ACTIONS = MODULES + SIGNATURES + ["stop"]

def valid_actions(partial):
    return MODULES if not partial else SIGNATURES if len(partial)==1 else ["stop"]

# 8. GPT-2 Policy Network Initialization
Each new run loads the same immutable pretrained GPT-2 revision and a fresh Adam optimizer; no checkpoint is searched for implicitly. Evaluation mode disables the pretrained dropout, matching the released notebook; autograd remains enabled during updates. Only one policy is held by the experiment session. Release an old session before starting another K.

Adam uses `foreach=False` to reduce temporary optimizer memory while retaining its mathematical update and default betas/epsilon. Initial parameter hashes are persisted; cross-K warm starts are prohibited.

In [ ]:
def parameter_digest(model):
    digest = hashlib.sha256()
    for name, parameter in model.named_parameters():
        digest.update(name.encode()); digest.update(parameter.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()

def initialize_policy(cfg):
    if cfg["policy_device"] == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA policy requested but unavailable. Select the GPU kernel or explicitly use CPU")
    seed_everything(cfg["seed"])
    try:
        tokenizer = GPT2Tokenizer.from_pretrained(cfg["policy_model"],revision=cfg["policy_revision"])
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.truncation_side = "left"  # preserves Pipeline suffix if a custom long prompt exceeds 1024
        model = GPT2LMHeadModel.from_pretrained(cfg["policy_model"],revision=cfg["policy_revision"])
    except Exception as exc:
        raise RuntimeError("GPT-2 files unavailable: "+str(exc)) from exc
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model.to(cfg["policy_device"]).eval()
    optimizer = torch.optim.Adam(model.parameters(),lr=cfg["learning_rate"],foreach=False)
    first_tokens = {action:tokenizer.encode(action,add_special_tokens=False)[0] for action in ACTIONS}
    initial_digest = parameter_digest(model)
    return model, tokenizer, optimizer, first_tokens, initial_digest

def release_experiment(session):
    if session is not None:
        for key in ("policy","optimizer","tokenizer"):
            session.pop(key,None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 9. Policy Action Generation
Scores are the first GPT-2 token logits of valid action **entries**. Fifteen signatures share nine first-token IDs: collision entries remain separate and cannot learn relative preference within a collision group. `log_softmax` preserves the sampled categorical probabilities; no full-string scoring is introduced. The same prompt and partial-action state are used for rollout and differentiable replay.
Action diagnostics record all entry probabilities and entropy during rollout. Same-question probes enumerate both module-specific signature distributions before and after each update without sampling or changing the RNG. Signature entropy in episode rows is the module-probability-weighted conditional entropy (nats).


In [ ]:
def action_log_probs(model, tokenizer, first_tokens, prompt, partial, cfg):
    choices = valid_actions(partial)
    text = f"Prompt: {prompt} Pipeline: {' '.join(partial)}"
    encoded = tokenizer(text,return_tensors="pt",padding=True,truncation=True,
                        max_length=min(cfg["max_policy_tokens"],model.config.n_positions))
    device = next(model.parameters()).device
    encoded = {key:value.to(device) for key,value in encoded.items()}
    logits = model(**encoded,use_cache=False).logits[0,-1]
    ids = torch.tensor([first_tokens[action] for action in choices],device=device)
    return choices, torch.log_softmax(logits[ids],dim=0)

def action_statistics(choices, scores):
    probabilities = scores.detach().double().exp()
    return dict(probabilities=dict(zip(choices,probabilities.cpu().tolist())),
                entropy_nats=float(-(probabilities*scores.detach().double()).sum()))


def probe_policy(session, prompt):
    # Enumerate both signature conditionals; no draws, graphs, or RNG changes.
    model, cfg = session["policy"], session["config"]
    previous_mode = model.training
    model.eval()
    try:
        with torch.no_grad():
            choices, scores = action_log_probs(model,session["tokenizer"],session["first_tokens"],prompt,[],cfg)
            modules = action_statistics(choices,scores)
            signatures = {}
            for module in MODULES:
                choices, scores = action_log_probs(model,session["tokenizer"],session["first_tokens"],prompt,[module],cfg)
                signatures[module] = action_statistics(choices,scores)
        expected = sum(modules["probabilities"][m]*signatures[m]["entropy_nats"] for m in MODULES)
        return dict(module=modules,signature_given_module=signatures,
                    expected_signature_entropy_nats=expected,
                    joint_pipeline_entropy_nats=modules["entropy_nats"]+expected,
                    entropy_units="nats",scope="fixed current training question; deterministic stop excluded")
    finally:
        model.train(previous_mode)

def sample_actions(model, tokenizer, first_tokens, prompt, cfg):
    partial, log_probs, diagnostics = [], [], []
    with torch.no_grad():
        for _ in range(cfg["max_pipeline_steps"]):
            choices, scores = action_log_probs(model,tokenizer,first_tokens,prompt,partial,cfg)
            index = int(torch.multinomial(scores.exp(),1).item())
            action = choices[index]
            diagnostics.append(dict(state_actions=list(partial),selected_action=action,
                                    selected_probability=float(scores[index].double().exp()),
                                    **action_statistics(choices,scores)))
            partial.append(action); log_probs.append(float(scores[index]))
            if action == "stop":
                break
    return dict(actions=partial,log_probs=log_probs,action_diagnostics=diagnostics)

def replay_log_probs(model, tokenizer, first_tokens, prompt, trajectory, cfg):
    partial, log_probs = [], []
    for action in trajectory["actions"]:
        choices, scores = action_log_probs(model,tokenizer,first_tokens,prompt,partial,cfg)
        if action not in choices:
            raise ValueError("Trajectory action violates released action grammar")
        log_probs.append(scores[choices.index(action)])
        partial.append(action)
    return log_probs

def action_collision_report(first_tokens):
    groups = {}
    for signature in SIGNATURES:
        groups.setdefault(first_tokens[signature],[]).append(signature)
    return {token:actions for token,actions in groups.items() if len(actions)>1}

# 10. DSPy Pipeline Generation
`generate_pipeline()` always receives the intended policy explicitly. This corrects the original evaluator's accidental use of a global policy. Generated actions and signatures are logged for every attempted candidate.

In [ ]:
def generate_pipeline(model, tokenizer, first_tokens, prompt, cfg):
    trajectory = sample_actions(model,tokenizer,first_tokens,prompt,cfg)
    actions = trajectory["actions"]
    trajectory["module"] = actions[0] if actions else None
    trajectory["signature"] = actions[1] if len(actions)>1 else None
    return trajectory

# 11. DSPy Pipeline Execution
Candidates execute sequentially. The original square-bracket instruction and dynamic signature input/output fields are preserved. Exceptions become logged failed candidates with reward zero; candidate, judge, and infrastructure errors are distinguished. A service failure still biases that group, so inspect failure logs before interpreting comparisons.

In [ ]:
def execute_pipeline(trajectory, prompt, lm):
    started = time.perf_counter()
    try:
        module, signature = trajectory["module"],trajectory["signature"]
        if module not in MODULES or signature not in SIGNATURES:
            raise ValueError("Invalid pipeline module or signature")
        input_field, output_field = [part.strip() for part in signature.split("->")]
        program = dspy.ChainOfThought(signature) if module == "CoT" else dspy.Predict(signature)
        instruction = "system instruction: Must give your final answer in square brackets without fail! e.g., [final answer] like this: [5] \n prompt: "
        with dspy.context(lm=lm):
            response = program(**{input_field:instruction+prompt})
        output = response.get(output_field)
        if output is None:
            raise ValueError("DSPy returned no signature output")
        return dict(output=str(output),success=True,error=None,seconds=time.perf_counter()-started)
    except Exception as exc:
        return dict(output=None,success=False,error=f"{type(exc).__name__}: {exc}",seconds=time.perf_counter()-started)

# 12. Reward Calculation
Correct numeric comparison supports negative numbers, decimals and thousands separators. Prefer the final square-bracket answer, then `####`, then the last numeric span (the released extraction convention). Last-number extraction can still mistake a reasoning number for a final answer; this limitation is recorded. `5` never matches `50` by substring.

Main reward remains exact-match 1, otherwise the same execution LLM judges fractional correctness in [0,1]. Parsing rejects nonfinite values and malformed judge responses. There is no formatting bonus. These normalization fixes change exact-match rewards relative to the broken original. Strict numeric EM and semantic fallback are logged separately.
Semantic fallbacks also preserve the exact judge prompt, returned raw strings, provider response when available, and whitelisted effective request parameters. This is observational logging only: the reward prompt, parsing and score remain unchanged. Authentication keys are excluded.


In [ ]:
NUMBER_PATTERN = re.compile(r"[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?|[-+]?\.\d+")

def extract_answer(response):
    if response is None:
        return None
    text = str(response).strip()
    brackets = re.findall(r"\[([^\[\]]+)\]",text)
    if brackets:
        text = brackets[-1].strip()
        matches = list(NUMBER_PATTERN.finditer(text))
        if len(matches)!=1:
            return None
        return canonical_number(matches[0].group())
    if "####" in text:
        text = text.rsplit("####",1)[1].strip()
    matches = list(NUMBER_PATTERN.finditer(text))
    return canonical_number(matches[-1].group()) if matches else None

def compare_answers(response, target):
    prediction = extract_answer(response)
    truth = canonical_number(target)
    return prediction is not None and truth is not None and prediction == truth

def parse_judge_score(text):
    # Accept a complete number or final [number], never substring matching or NaN/Inf.
    value = str(text).strip()
    bracket = re.fullmatch(r"\s*\[\s*([^\[\]]+)\s*\]\s*",value)
    if bracket:
        value = bracket.group(1).strip()
    if re.fullmatch(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)",value) is None:
        raise ValueError("Judge did not return a single finite numeric score")
    score = float(value)
    if not math.isfinite(score):
        raise ValueError("Nonfinite judge score")
    return min(1.0,max(0.0,score))

def compute_reward(prompt, response, target, lm):
    exact = compare_answers(response,target)
    if response is None:
        return dict(reward=0.0,numeric_exact_match=False,semantic_used=False,judge_score=None,judge_error=None)
    if exact:
        return dict(reward=1.0,numeric_exact_match=True,semantic_used=False,judge_score=None,judge_error=None)
    judge_prompt = ("Evaluate whether the following response correctly answers the prompt based on the ground truth and give your final score in the square brackets[final score]. only the value in the [] should in your response and nothing else.\n"
                    f"Prompt: {prompt}\nResponse: {response}\nGround Truth: {target}\n"
                    "Return a score in range between 1.0 to 0.0 if the response is correct or partially correct (matches or is equivalent to the ground truth), or 0.0 if incorrect.\nfinal score:[]")
    trace = dict(prompt=judge_prompt,raw_responses=[],inference=judge_inference_parameters(lm),provider_calls=[])
    event_start = len(getattr(lm,"events",[]))
    previous_capture = getattr(lm,"capture_semantic_judge",False)
    lm.capture_semantic_judge = True
    try:
        outputs = lm(judge_prompt)
        trace["raw_responses"] = copy.deepcopy(outputs)
        score = parse_judge_score(outputs[0])
        return dict(reward=score,numeric_exact_match=False,semantic_used=True,judge_score=score,judge_error=None,semantic_judge=trace)
    except Exception as exc:
        return dict(reward=0.0,numeric_exact_match=False,semantic_used=True,judge_score=None,
                    judge_error=f"{type(exc).__name__}: {exc}",semantic_judge=trace)
    finally:
        lm.capture_semantic_judge = previous_capture
        trace["provider_calls"] = [copy.deepcopy(event) for event in getattr(lm,"events",[])[event_start:]
                                   if "judge_request" in event]

# 13. Figure 2 GRPO Loss
$$\bar r=\frac1K\sum_i r_i,\quad A_i=r_i-\bar r,\quad \mathcal L=-\sum_i\sum_t\log\pi_\theta(a_{i,t}|s_{i,t})A_i.$$
No division by K, clipping, entropy, standard-deviation normalization, GAE, KL or gradient clipping. Larger K can change gradient magnitudes; log gradient norms rather than silently renormalize.

Rollouts use no-grad; each trajectory's differentiable log probabilities are replayed and backpropagated sequentially before **one** Adam step. Parameters stay fixed during replay. Finite equal-reward groups still call Adam.step, preserving the released behavior: momentum can move parameters even with zero current gradients.

In [ ]:
def group_advantages(rewards):
    values = np.asarray(rewards,dtype=np.float64)
    if len(values)==0 or not np.isfinite(values).all():
        raise ValueError("GRPO requires a nonempty finite group")
    return float(values.mean()), values-values.mean()

def figure2_loss(log_prob_trajectories, advantages):
    if len(log_prob_trajectories)!=len(advantages):
        raise ValueError("One advantage per candidate is required")
    return -sum(sum(log_probs)*float(advantage) for log_probs,advantage in zip(log_prob_trajectories,advantages))

def update_policy(session, prompt, trajectories, rewards):
    if session.get("evaluation_only",False):
        raise RuntimeError("Evaluation-only checkpoint: use explicit resume before updating")
    model, optimizer = session["policy"],session["optimizer"]
    cfg = session["config"]
    if len(trajectories)!=cfg["k"] or len(rewards)!=cfg["k"]:
        raise ValueError("Episode must contain exactly K candidate rewards")
    mean, advantages = group_advantages(rewards)
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    model.eval()
    for trajectory, advantage in zip(trajectories,advantages):
        scores = replay_log_probs(model,session["tokenizer"],session["first_tokens"],prompt,trajectory,cfg)
        replay_values = [float(score.detach()) for score in scores]
        if not np.allclose(replay_values,trajectory["log_probs"],rtol=2e-5,atol=2e-5):
            optimizer.zero_grad(set_to_none=True)
            raise RuntimeError("Replay differs from rollout; parameters/mode/state changed before update")
        loss = figure2_loss([scores],[advantage])
        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite Figure 2 loss; no optimizer step permitted")
        total_loss += float(loss.detach())
        loss.backward()
        del loss, scores
    norm_sq = sum(float(parameter.grad.detach().double().square().sum())
                  for parameter in model.parameters() if parameter.grad is not None)
    norm = math.sqrt(norm_sq)
    if not math.isfinite(norm):
        optimizer.zero_grad(set_to_none=True)
        raise RuntimeError("Nonfinite gradient; no optimizer step permitted")
    optimizer.step()
    session.pop("loaded_checkpoint",None)
    optimizer.zero_grad(set_to_none=True)
    return dict(group_mean=mean,advantages=advantages.tolist(),loss=total_loss,gradient_norm=norm,
                zero_advantage_group=bool(np.all(advantages==0)),optimizer_step=True)

# 14. GRPO Training Loop
Every episode uses one scheduled question and attempts exactly K candidates sequentially. Validation has isolated RNG state and uses no test questions. Periodic checkpoints, validation checkpoints and safe between-episode interruption preserve completed work.

Inside-episode interruption requires restoring the latest durable checkpoint. Unsaved optimizer work and statistical rows roll back together; external LLM work cannot be undone and remains in the attempt journal. Resume starts at the next **durably committed** episode. Set checkpoint interval to 1 for minimum rollback, at substantial disk-I/O cost.

In [ ]:
def initialize_experiment(cfg, create_run=True):
    cfg, datasets = load_gsm8k(cfg)
    data = prepare_selection(cfg,datasets)
    lm, llm_metadata = initialize_ollama(cfg)
    model, tokenizer, optimizer, first_tokens, digest = initialize_policy(cfg)
    session = dict(config=cfg,data=data,lm=lm,llm_metadata=llm_metadata,policy=model,tokenizer=tokenizer,
                   optimizer=optimizer,first_tokens=first_tokens,initial_policy_digest=digest,
                   state=dict(episode=0,updates=0,train=[],validation=[],tests=[],best=None,seen=[],
                              cumulative_training_seconds=0.0),run_dir=None,in_episode=False)
    if create_run:
        run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")+"-"+uuid.uuid4().hex[:8]
        root = Path(cfg["results_root"])/"gsm8k"/f"k{cfg['k']}"/f"seed{cfg['seed']}"/cfg["preset"]/run_id
        root.mkdir(parents=True,exist_ok=False); (root/"checkpoints").mkdir()
        session["run_dir"] = root
        write_json(root/"config.json",dict(config=cfg,selection_id=data["selection_id"],
                   selection_path=data["selection_path"],initial_policy_digest=digest,
                   hardware=hardware_metadata(),execution_model=llm_metadata))
    return session

def append_journal(session,event):
    if session["run_dir"] is None:
        return
    with (session["run_dir"]/"attempts.jsonl").open("a",encoding="utf-8") as stream:
        stream.write(json.dumps(event,allow_nan=False)+"\n"); stream.flush(); os.fsync(stream.fileno())

def train_grpo(session):
    if session.get("evaluation_only",False):
        raise RuntimeError("Evaluation-only checkpoint: use explicit resume before training")
    cfg,state=session["config"],session["state"]
    if session["run_dir"] is None:
        raise RuntimeError("Training requires an explicitly created or resumed run")
    if session["in_episode"]:
        raise RuntimeError("Interrupted episode: restore latest durable checkpoint")
    validated={row["episode"] for row in state["validation"]}
    if state["episode"]>=cfg["num_episodes"] and cfg["num_episodes"] in validated:
        raise RuntimeError("Run completed; create a new run or evaluate its checkpoint")
    pending_validation=(state["episode"]==0 or state["episode"]%cfg["validation_interval"]==0 or state["episode"]==cfg["num_episodes"]) and state["episode"] not in validated
    if pending_validation:
        validation=evaluate_module(session,"validation",episode=state["episode"])
        state["validation"].append(validation); save_checkpoint(session,validation=validation)
    elif not (session["run_dir"]/"latest.json").exists():
        save_checkpoint(session)
    if state["episode"]==cfg["num_episodes"]:
        write_summary(session)
        return session
    try:
        while state["episode"]<cfg["num_episodes"]:
            episode=state["episode"]+1
            offset=session["data"]["selection"]["episode_pool_offsets"][episode-1]
            example=session["data"]["subsets"]["train"][offset]
            session["in_episode"]=True
            if next(session["policy"].parameters()).is_cuda:
                torch.cuda.reset_peak_memory_stats()
            begin=time.perf_counter(); call_start=len(session["lm"].events)
            policy_before = probe_policy(session,example["question"])
            trajectories,results,reward_details=[],[],[]
            for candidate in range(cfg["k"]):
                candidate_call_start=len(session["lm"].events)
                trajectory=generate_pipeline(session["policy"],session["tokenizer"],session["first_tokens"],example["question"],cfg)
                result=execute_pipeline(trajectory,example["question"],session["lm"])
                reward=compute_reward(example["question"],result["output"],example["answer"],session["lm"])
                trajectories.append(trajectory); results.append(result); reward_details.append(reward)
                append_journal(session,dict(kind="training_attempt",episode=episode,candidate=candidate,
                               question_id=example["id"],execution=result,reward=reward,
                               resources=call_statistics(session["lm"].events[candidate_call_start:]),
                               timestamp=datetime.now(timezone.utc).isoformat()))
            rewards=[detail["reward"] for detail in reward_details]
            update=update_policy(session,example["question"],trajectories,rewards)
            policy_after = probe_policy(session,example["question"])
            seconds=time.perf_counter()-begin
            seen=set(state["seen"]); seen.add(example["id"])
            calls=call_statistics(session["lm"].events[call_start:])
            row=dict(dataset=cfg["dataset"],k=cfg["k"],seed=cfg["seed"],episode=episode,question_id=example["id"],
                     question_pool_offset=offset,actions=[t["actions"] for t in trajectories],
                     signatures=[t["signature"] for t in trajectories],rewards=rewards,
                     unique_pipeline_count=len({tuple(t["actions"]) for t in trajectories}),
                     all_candidate_actions_identical=len({tuple(t["actions"]) for t in trajectories})==1,
                     zero_reward_variance=bool(np.var(rewards)==0),
                     policy_diagnostics=dict(before_update=policy_before,after_update=policy_after),
                     module_entropy_nats=policy_before["module"]["entropy_nats"],
                     signature_entropy_nats=policy_before["expected_signature_entropy_nats"],
                     grpo_loss_scientific=f"{update['loss']:.9e}",
                     gradient_norm_scientific=f"{update['gradient_norm']:.9e}",
                     mean_group_reward=update["group_mean"],reward_std=float(np.std(rewards)),reward_variance=float(np.var(rewards)),
                     advantages=update["advantages"],zero_advantage_group=update["zero_advantage_group"],
                     grpo_loss=update["loss"],gradient_norm=update["gradient_norm"],optimizer_update_count=state["updates"]+1,
                     unique_training_questions=len(seen),attempted_pipeline_evaluations=cfg["k"],
                     successful_pipeline_evaluations=sum(result["success"] for result in results),
                     failed_pipeline_evaluations=sum(not result["success"] for result in results),
                     episode_seconds=seconds,cumulative_training_seconds=state["cumulative_training_seconds"]+seconds,
                     average_pipeline_execution_latency=float(np.mean([result["seconds"] for result in results])),
                     pipeline_execution_seconds=sum(result["seconds"] for result in results),
                     candidate_details=[dict(trajectory=t,execution=r,reward=w) for t,r,w in zip(trajectories,results,reward_details)],
                     out_of_memory_failures=sum("out of memory" in (result["error"] or "").lower() for result in results),
                     execution_failures=[result["error"] for result in results if result["error"]],
                     judge_failures=[detail["judge_error"] for detail in reward_details if detail["judge_error"]],
                     ollama_memory=ollama_snapshot(cfg),**calls,**gpu_statistics(session["policy"]))
            state["train"].append(row); state["episode"]=episode; state["updates"]+=1
            state["seen"]=sorted(seen); state["cumulative_training_seconds"]+=seconds
            session["in_episode"]=False
            synchronize_logs(session)
            validation=None
            if episode % cfg["validation_interval"]==0 or episode==cfg["num_episodes"]:
                validation=evaluate_module(session,"validation",episode=episode)
                state["validation"].append(validation)
            if validation is not None or episode % cfg["checkpoint_interval"]==0 or episode==cfg["num_episodes"]:
                save_checkpoint(session,validation=validation)
            write_summary(session)
            print(f"K={cfg['k']} episode={episode}/{cfg['num_episodes']} "
                  f"reward={row['mean_group_reward']:.3f} unique={row['unique_pipeline_count']}/{cfg['k']} "
                  f"H_module={row['module_entropy_nats']:.3e} H_signature={row['signature_entropy_nats']:.3e} "
                  f"loss={row['grpo_loss']:.6e} grad={row['gradient_norm']:.6e}",flush=True)
    except BaseException as exc:
        append_journal(session,dict(kind="interruption",episode=state["episode"]+int(session["in_episode"]),
                       error=f"{type(exc).__name__}: {exc}",in_episode=session["in_episode"],
                       out_of_memory="out of memory" in str(exc).lower(),
                       gpu_memory=gpu_statistics(session["policy"]),
                       timestamp=datetime.now(timezone.utc).isoformat()))
        if not session["in_episode"]:
            save_checkpoint(session)
            write_summary(session)
        else:
            print("Episode interrupted. Resume the latest.json checkpoint; incomplete work was not committed.")
        raise
    return session

# Definitions only: training is explicitly invoked in section 21.

# 15. Checkpoint Management
Immutable filenames prevent accidental checkpoint overwrite; atomic latest/best pointers stay in the explicit run. Select best by highest validation numeric EM; **earliest episode wins ties**. Every validation state, including episode 0, is checkpointed. Resume only the latest durable checkpoint; historical checkpoints are available for evaluation.

Checkpoints contain policy, Adam, complete config, all RNG states, selection/schedule, metrics and execution identity. Load only your own trusted checkpoints (`weights_only=False` permits Python/NumPy RNG restoration). Final-test results never select checkpoints.

In [ ]:
def checkpoint_payload(session):
    if session.get("evaluation_only",False):
        raise RuntimeError("Do not save training checkpoints from an evaluation-only session")
    if session["in_episode"]:
        raise RuntimeError("Cannot checkpoint an incomplete episode")
    return dict(format_version=1,config=copy.deepcopy(session["config"]),
                policy=session["policy"].state_dict(),optimizer=session["optimizer"].state_dict(),
                state=copy.deepcopy(session["state"]),rng=rng_state(),
                selection=copy.deepcopy(session["data"]["selection"]),selection_id=session["data"]["selection_id"],
                initial_policy_digest=session["initial_policy_digest"],execution_model=session["llm_metadata"],
                hardware=hardware_metadata(),run_dir=str(session["run_dir"]),
                saved_at=datetime.now(timezone.utc).isoformat())

def atomic_torch_save(path,payload):
    path=Path(path)
    if path.exists():
        raise FileExistsError(f"Refusing checkpoint overwrite: {path}")
    temporary=path.with_name(path.name+"."+uuid.uuid4().hex+".tmp")
    with temporary.open("wb") as stream:
        torch.save(payload,stream); stream.flush(); os.fsync(stream.fileno())
    os.link(temporary,path); temporary.unlink()

def save_checkpoint(session,validation=None):
    state=session["state"]; root=session["run_dir"]
    if root is None:
        raise RuntimeError("No active run directory")
    path=root/"checkpoints"/f"episode_{state['episode']:04d}_{uuid.uuid4().hex[:8]}.pt"
    old_best=state["best"]
    improved=validation is not None and (old_best is None or validation["numeric_exact_match_accuracy"]>old_best["score"])
    if improved:
        state["best"]=dict(score=validation["numeric_exact_match_accuracy"],episode=state["episode"],checkpoint=str(path))
    try:
        atomic_torch_save(path,checkpoint_payload(session))
    except BaseException:
        state["best"]=old_best
        raise
    write_json(root/"latest.json",dict(checkpoint=str(path),episode=state["episode"]),overwrite=True)
    if improved:
        write_json(root/"best.json",state["best"],overwrite=True)
    synchronize_logs(session)
    return path

def compatible_config(expected,actual):
    ignored={"results_root"}
    differing=[key for key in set(expected)|set(actual) if key not in ignored and expected.get(key)!=actual.get(key)]
    if differing:
        raise ValueError("Checkpoint configuration mismatch: "+", ".join(sorted(differing)))

def load_checkpoint_into(session,path,for_resume=False):
    path=Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Selected checkpoint not found: {path}")
    payload=torch.load(path,map_location="cpu",weights_only=False)
    if payload.get("format_version")!=1:
        raise ValueError("Unsupported checkpoint format")
    compatible_config(session["config"],payload["config"])
    if payload["selection_id"]!=session["data"]["selection_id"] or payload["selection"]!=session["data"]["selection"]:
        raise ValueError("Checkpoint dataset/schedule mismatch")
    if payload["initial_policy_digest"]!=session["initial_policy_digest"]:
        raise ValueError("Pretrained GPT-2 initial parameters differ")
    old_model=payload["execution_model"].get("model",{})
    new_model=session["llm_metadata"].get("model",{})
    if old_model.get("digest")!=new_model.get("digest"):
        raise ValueError("Ollama model digest changed; refuse controlled resume/evaluation")
    if payload["hardware"]["packages"]!=VERSIONS or payload["execution_model"].get("version")!=session["llm_metadata"].get("version"):
        raise ValueError("Dependency/Ollama versions changed; refuse controlled checkpoint loading")
    root=Path(payload["run_dir"])
    if for_resume:
        if not root.exists():
            raise FileNotFoundError("Original run directory unavailable")
        latest=json.loads((root/"latest.json").read_text(encoding="utf-8"))
        if Path(latest["checkpoint"]).resolve()!=path.resolve():
            raise ValueError("Resume only latest durable checkpoint")
        if payload["state"]["episode"]>=payload["config"]["num_episodes"] and any(row["episode"]==payload["config"]["num_episodes"] for row in payload["state"]["validation"]):
            raise ValueError("Run completed; cannot resume as a longer/different preset")
    session["policy"].load_state_dict(payload["policy"]); session["policy"].eval()
    if for_resume:
        session["optimizer"].load_state_dict(payload["optimizer"])
    session["state"]=payload["state"]; session["run_dir"]=root; session["in_episode"]=False
    session["loaded_checkpoint"]=str(path)
    session["evaluation_only"]=not for_resume
    restore_rng(payload["rng"])
    if for_resume:
        synchronize_logs(session)
        if session["state"]["best"] is not None:
            write_json(root/"best.json",session["state"]["best"],overwrite=True)
        write_summary(session)
        append_journal(session,dict(kind="resume",next_episode=session["state"]["episode"]+1,
                                   checkpoint=str(path),timestamp=datetime.now(timezone.utc).isoformat()))
    del payload; gc.collect()
    return session

def load_selected_experiment(path,cfg,for_resume=False):
    session=initialize_experiment(cfg,create_run=False)
    return load_checkpoint_into(session,path,for_resume=for_resume)

# 16. Validation
One sampled pipeline per fixed validation question, no gradients or optimizer updates. A fixed split-specific policy RNG seed is reused at each checkpoint, then the training RNG is restored. Llama uses fixed seed/temperature; version/device changes can still affect outputs. Strict numeric EM, fractional semantic scores and failures remain distinct.

In [ ]:
def evaluate_module(session,split="validation",episode=None,examples=None):
    if split not in ("validation","test","full_test"):
        raise ValueError("Evaluation must use validation/test, never training")
    if examples is None:
        if split=="full_test":
            raise ValueError("Full test requires explicit examples")
        examples=session["data"]["subsets"][split]
    if not examples:
        raise ValueError("Empty evaluation set")
    train_ids=set(session["data"]["selection"]["ids"]["train"])
    if any(example["id"] in train_ids for example in examples):
        raise ValueError("Training examples cannot enter held-out evaluation")
    cfg=session["config"]; model=session["policy"]
    old_mode=model.training; model.eval()
    call_start=len(session["lm"].events); begin=time.perf_counter(); records=[]
    seed=cfg["seed"]+cfg["evaluation_seed_offset"]+(0 if split=="validation" else 1)
    try:
        with isolated_rng(seed),torch.no_grad():
            for example in examples:
                question_call_start=len(session["lm"].events)
                trajectory=generate_pipeline(model,session["tokenizer"],session["first_tokens"],example["question"],cfg)
                execution=execute_pipeline(trajectory,example["question"],session["lm"])
                reward=compute_reward(example["question"],execution["output"],example["answer"],session["lm"])
                append_journal(session,dict(kind="evaluation_attempt",split=split,episode=session["state"]["episode"],
                               question_id=example["id"],execution=execution,reward=reward,
                               resources=call_statistics(session["lm"].events[question_call_start:]),
                               timestamp=datetime.now(timezone.utc).isoformat()))
                records.append(dict(question_id=example["id"],target=example["answer"],prediction=extract_answer(execution["output"]),
                                    trajectory=trajectory,execution=execution,**reward))
    finally:
        model.train(old_mode)
    exact=sum(record["numeric_exact_match"] for record in records)
    scores=[record["judge_score"] for record in records if record["judge_score"] is not None]
    return dict(dataset=cfg["dataset"],k=cfg["k"],seed=cfg["seed"],split=split,
                episode=session["state"]["episode"] if episode is None else episode,
                selected_checkpoint=session.get("loaded_checkpoint"),questions=len(records),
                question_ids=[r["question_id"] for r in records],numeric_exact_match_accuracy=exact/len(records),
                semantic_fallback_count=sum(record["semantic_used"] for record in records),
                semantic_fallback_mean_score=float(np.mean(scores)) if scores else None,
                semantic_full_credit_count=sum(score==1.0 for score in scores),
                mean_reward=float(np.mean([r["reward"] for r in records])),
                attempted_pipeline_evaluations=len(records),successful_pipeline_evaluations=sum(r["execution"]["success"] for r in records),
                failed_pipeline_evaluations=sum(not r["execution"]["success"] for r in records),
                cumulative_training_pipeline_evaluations=sum(r["attempted_pipeline_evaluations"] for r in session["state"]["train"]),
                seconds=time.perf_counter()-begin,records=records,**call_statistics(session["lm"].events[call_start:]))

def validate_selected(session):
    if "loaded_checkpoint" not in session:
        raise RuntimeError("Load intended checkpoint before standalone validation")
    result=evaluate_module(session,"validation"); save_evaluation_artifact(session,result)
    refresh_test_summary(session["run_dir"])
    return result

# 17. Final Testing
Final testing explicitly loads the validation-selected checkpoint and uses the fixed official-test subset. It never selects checkpoints. Full official-test evaluation is opt-in and can be expensive. Test artifacts do not mutate historical training checkpoints.

In [ ]:
def save_evaluation_artifact(session,result):
    path=session["run_dir"]/f"{result['split']}_{uuid.uuid4().hex[:8]}.json"
    write_json(path,result)
    return path

def test_model(session,full_official=False):
    if "loaded_checkpoint" not in session:
        raise RuntimeError("Final testing requires an explicitly loaded checkpoint")
    training_summary=json.loads((session["run_dir"]/"summary.json").read_text(encoding="utf-8"))
    if not training_summary["completed"]:
        raise RuntimeError("Finish training before final testing")
    best_path=session["run_dir"]/"best.json"
    if not best_path.exists():
        raise RuntimeError("No validation-selected checkpoint")
    best=json.loads(best_path.read_text(encoding="utf-8"))
    if Path(session["loaded_checkpoint"]).resolve()!=Path(best["checkpoint"]).resolve():
        raise ValueError("Load validation-selected best checkpoint before final testing")
    if full_official:
        official=session["data"]["full_test"]
        examples=[dict(id=f"gsm8k:test:{i}",split="test",index=i,question=item["question"],answer=canonical_target(item["answer"]),rationale=item["answer"])
                  for i,item in enumerate(official)]
        result=evaluate_module(session,"full_test",examples=examples)
    else:
        result=evaluate_module(session,"test")
    result["checkpoint_selection_rule"]="maximum validation numeric EM; earliest episode on ties"
    path=save_evaluation_artifact(session,result)
    refresh_test_summary(session["run_dir"])
    print("Held-out numeric EM:",result["numeric_exact_match_accuracy"],"artifact:",path)
    return result

# 18. Metrics and Statistics
JSONL holds episode and validation rows. `attempts.jsonl` retains failed/discarded external work; effective training rows are reconciled to restored checkpoint state. Training time excludes validation and checkpoint I/O; total wall time is unavailable. Token totals are unavailable if any relevant call omits usage. Ollama memory is a server snapshot, distinct from PyTorch GPU allocator peaks.
Completed summaries also aggregate measured GPU peaks, training/evaluation failures and LM time. Journal totals retain observed calls/tokens/time after rollback; missing or incomplete journals stay unavailable. Interrupted calls without completed journal entries are outside this scope.


In [ ]:
def call_statistics(events):
    def total(key):
        return sum(event[key] for event in events) if all(event.get(key) is not None for event in events) else None
    return dict(execution_llm_calls=len(events),physical_http_requests=None,prompt_tokens=total("prompt_tokens"),
                generated_tokens=total("generated_tokens"),llm_seconds=sum(e["seconds"] for e in events),
                llm_failed_calls=sum(e["error"] is not None for e in events),
                call_count_definition="DSPy LM.forward attempts, including semantic judge and adapter retries")

def gpu_statistics(model):
    if not next(model.parameters()).is_cuda:
        return dict(peak_allocated_gpu_bytes=None,peak_reserved_gpu_bytes=None)
    return dict(peak_allocated_gpu_bytes=torch.cuda.max_memory_allocated(),peak_reserved_gpu_bytes=torch.cuda.max_memory_reserved())

def write_jsonl(path,rows):
    path=Path(path); temporary=path.with_name(path.name+"."+uuid.uuid4().hex+".tmp")
    with temporary.open("w",encoding="utf-8") as stream:
        for row in rows:
            stream.write(json.dumps(row,ensure_ascii=False,allow_nan=False)+"\n")
        stream.flush(); os.fsync(stream.fileno())
    os.replace(temporary,path)

def synchronize_logs(session):
    if session["run_dir"] is not None:
        write_jsonl(session["run_dir"]/"training.jsonl",session["state"]["train"])
        write_jsonl(session["run_dir"]/"validation.jsonl",session["state"]["validation"])

def recorded_resource_totals(train, evaluations, journal):
    """Aggregate recorded artifacts only; missing measurements remain None."""
    def total(rows, key):
        values=[row.get(key) for row in rows]
        return sum(values) if all(value is not None for value in values) else None
    def peak(key):
        values=[row.get(key) for row in train]
        return max(values) if values and all(value is not None for value in values) else None
    attempts=[row for row in journal if row.get("kind") in ("training_attempt","evaluation_attempt")]
    effective_attempts=sum(row["attempted_pipeline_evaluations"] for row in train+evaluations)
    complete=len(attempts)>=effective_attempts and all("resources" in row for row in attempts)
    resources=[row["resources"] for row in attempts] if complete else []
    result=dict(peak_allocated_gpu_bytes=peak("peak_allocated_gpu_bytes"),
                peak_reserved_gpu_bytes=peak("peak_reserved_gpu_bytes"),
                training_out_of_memory_failures=total(train,"out_of_memory_failures"),
                training_judge_failures=sum(len(row.get("judge_failures",[])) for row in train),
                training_llm_failed_calls=total(train,"llm_failed_calls"),
                training_llm_seconds=total(train,"llm_seconds"),
                evaluation_successful_pipeline_evaluations=total(evaluations,"successful_pipeline_evaluations"),
                evaluation_failed_pipeline_evaluations=total(evaluations,"failed_pipeline_evaluations"),
                evaluation_llm_failed_calls=total(evaluations,"llm_failed_calls"),
                evaluation_llm_seconds=total(evaluations,"llm_seconds"),
                resource_journal_complete=complete,
                observed_pipeline_attempts_including_rollback=len(attempts) if complete else None,
                journal_scope="completed journal entries; in-flight interrupted calls may be unavailable")
    for key in ("execution_llm_calls","prompt_tokens","generated_tokens","llm_seconds","llm_failed_calls"):
        label="llm_calls" if key=="execution_llm_calls" else key
        result["observed_"+label+"_including_rollback"]=total(resources,key) if complete else None
    return result

def experiment_summary(session):
    state,cfg=session["state"],session["config"]; rows=state["train"]
    attempts=sum(r["attempted_pipeline_evaluations"] for r in rows)
    latest_path=session["run_dir"]/"latest.json"
    durable=json.loads(latest_path.read_text(encoding="utf-8"))["episode"] if latest_path.exists() else 0
    tests=[]
    for path in sorted(session["run_dir"].glob("test_*.json")):
        tests.append(dict(path=str(path),**json.loads(path.read_text(encoding="utf-8"))))
    scores=[t["numeric_exact_match_accuracy"] for t in tests]
    final_accuracy=scores[0] if scores and all(score==scores[0] for score in scores) else None
    test_status="available" if scores else "not run"
    if scores and final_accuracy is None:
        test_status="repeated tests disagree; inspect artifacts, no best-score selection"
    def total(key):
        values=[r[key] for r in rows]
        return sum(values) if all(v is not None for v in values) else None
    evaluation_calls=sum(v["execution_llm_calls"] for v in state["validation"])+sum(t["execution_llm_calls"] for t in tests)
    return dict(config=cfg,selection_id=session["data"]["selection_id"],initial_policy_digest=session["initial_policy_digest"],
                execution_model_digest=session["llm_metadata"].get("model",{}).get("digest"),
                runtime_identity=dict(packages=VERSIONS,ollama_version=session["llm_metadata"].get("version")),run_dir=str(session["run_dir"]),
                completed=state["episode"]==cfg["num_episodes"] and durable==state["episode"] and any(row["episode"]==cfg["num_episodes"] for row in state["validation"]),
                episode=state["episode"],durable_episode=durable,optimizer_updates=state["updates"],
                unique_training_questions=len(state["seen"]),best=state["best"],test_status=test_status,
                final_held_out_accuracy=final_accuracy,training_time=state["cumulative_training_seconds"],
                attempted_pipeline_evaluations=attempts,successful_pipeline_evaluations=total("successful_pipeline_evaluations"),
                failed_pipeline_evaluations=total("failed_pipeline_evaluations"),execution_llm_calls=total("execution_llm_calls"),
                prompt_tokens=total("prompt_tokens"),generated_tokens=total("generated_tokens"),physical_http_requests=None,
                training_mean_reward=float(np.mean([r["mean_group_reward"] for r in rows])) if rows else None,
                training_reward_variance=float(np.var([x for r in rows for x in r["rewards"]])) if rows else None,
                zero_advantage_group_percentage=100*float(np.mean([r["zero_advantage_group"] for r in rows])) if rows else None,
                average_pipeline_execution_latency=sum(r["pipeline_execution_seconds"] for r in rows)/attempts if attempts else None,
                evaluation_llm_calls=evaluation_calls,
                evaluation_pipeline_evaluations=sum(v["attempted_pipeline_evaluations"] for v in state["validation"])+sum(t["attempted_pipeline_evaluations"] for t in tests),
                total_llm_calls_in_effective_history=total("execution_llm_calls")+evaluation_calls,total_wall_seconds=None,
                rollback_resource_accounting="attempts.jsonl retains discarded pipeline work")

def write_summary(session):
    summary=experiment_summary(session)
    write_json(session["run_dir"]/"summary.json",summary,overwrite=True)
    return refresh_test_summary(session["run_dir"])

def refresh_test_summary(root):
    root=Path(root)
    summary=json.loads((root/"summary.json").read_text(encoding="utf-8"))
    tests=[json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("test_*.json"))]
    extra_validation=[json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("validation_*.json"))]
    full_tests=[json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("full_test_*.json"))]
    scores=[result["numeric_exact_match_accuracy"] for result in tests]
    summary["final_held_out_accuracy"]=scores[0] if scores and all(score==scores[0] for score in scores) else None
    summary["test_status"]="available" if summary["final_held_out_accuracy"] is not None else "repeated tests disagree" if scores else "not run"
    validation=read_jsonl(root/"validation.jsonl")
    evaluations=validation+tests+extra_validation+full_tests
    summary["evaluation_llm_calls"]=sum(row["execution_llm_calls"] for row in evaluations)
    summary["evaluation_pipeline_evaluations"]=sum(row["attempted_pipeline_evaluations"] for row in evaluations)
    summary["evaluation_seconds"]=sum(row["seconds"] for row in evaluations)
    summary["total_llm_calls_in_effective_history"]=summary["execution_llm_calls"]+summary["evaluation_llm_calls"]
    journal=read_jsonl(root/"attempts.jsonl") if (root/"attempts.jsonl").exists() else []
    attempts=[row for row in journal if row["kind"] in ("training_attempt","evaluation_attempt")]
    summary["resource_journal_complete"]=all("resources" in row for row in attempts)
    summary["observed_pipeline_attempts_including_rollback"]=len(attempts) if summary["resource_journal_complete"] else None
    summary["observed_llm_calls_including_rollback"]=sum(row["resources"]["execution_llm_calls"] for row in attempts) if all("resources" in row for row in attempts) else None
    for key in ("prompt_tokens","generated_tokens"):
        values=[row[key] for row in evaluations]
        value=sum(values) if all(v is not None for v in values) else None
        summary["evaluation_"+key]=value
        summary["total_"+key]=summary[key]+value if summary[key] is not None and value is not None else None
    summary.update(recorded_resource_totals(read_jsonl(root/"training.jsonl"),evaluations,journal))
    write_json(root/"summary.json",summary,overwrite=True)
    return summary


# 19. Static-K Comparison
Compare completed runs only, with identical dataset selection, pretrained initialization, execution-model digest and non-K controls. Different presets/sizes are incompatible. Repeated runs remain separate; never choose the best final-test score. Artifact comparison does not initialize models or call Llama.
Comparisons verify complete episode/validation histories, candidate budgets, centered rewards, validation-selected checkpoint and held-out provenance before plotting. Configuration snapshots, persisted manifests and durable latest/best checkpoint pointers are required and checked. Duplicate run paths are rejected; independent repeats remain distinct.


In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

def verify_run_artifacts(root, summary):
    """Reject inconsistent histories before plotting; never rewrite source artifacts."""
    root=Path(root); cfg=summary["config"]
    train=read_jsonl(root/"training.jsonl"); validation=read_jsonl(root/"validation.jsonl")
    def require(condition, message):
        if not condition:
            raise ValueError(f"Inconsistent run artifacts at {root}: {message}")
    require(cfg["k"] in ALLOWED_K,"unsupported K")
    require(summary["episode"]==summary["durable_episode"]==cfg["num_episodes"],"incomplete durable history")
    require([row["episode"] for row in train]==list(range(1,cfg["num_episodes"]+1)),"missing/duplicate training episodes")
    cumulative=0; seen=set()
    for row in train:
        require((row["dataset"],row["k"],row["seed"])==(cfg["dataset"],cfg["k"],cfg["seed"]),"episode identity")
        require(row["attempted_pipeline_evaluations"]==cfg["k"],"candidate budget")
        require(row["successful_pipeline_evaluations"]+row["failed_pipeline_evaluations"]==cfg["k"],"candidate outcomes")
        require(len(row["rewards"])==len(row["advantages"])==len(row["actions"])==len(row["signatures"])==cfg["k"],"candidate records")
        mean, advantages=group_advantages(row["rewards"])
        require(np.isclose(row["mean_group_reward"],mean) and np.allclose(row["advantages"],advantages),"centered reward statistics")
        require(row["zero_advantage_group"]==bool(np.all(advantages==0)),"zero-advantage flag")
        cumulative+=cfg["k"]; seen.add(row["question_id"])
        require(row["unique_training_questions"]==len(seen),"unique question count")
        require(row["optimizer_update_count"]==row["episode"],"optimizer update count")
    require(summary["attempted_pipeline_evaluations"]==cumulative,"summary training budget")
    require(summary["optimizer_updates"]==len(train) and summary["unique_training_questions"]==len(seen),"summary updates/questions")
    for key in ("successful_pipeline_evaluations","failed_pipeline_evaluations","execution_llm_calls"):
        require(summary[key]==sum(row[key] for row in train),"summary "+key)
    require(np.isclose(summary["training_time"],sum(row["episode_seconds"] for row in train)),"summary training time")
    expected=[0]+[episode for episode in range(1,cfg["num_episodes"]+1)
                  if episode%cfg["validation_interval"]==0 or episode==cfg["num_episodes"]]
    require([row["episode"] for row in validation]==expected,"validation progression")
    validation_ids=validation[0]["question_ids"]
    require(len(validation_ids)==len(set(validation_ids))==cfg["validation_size"],"validation question IDs")
    for row in validation:
        require(row["question_ids"]==validation_ids and not seen.intersection(row["question_ids"]),"validation identity/leakage")
        require(row["questions"]==row["attempted_pipeline_evaluations"]==cfg["validation_size"],"validation pipeline budget")
        require(row["cumulative_training_pipeline_evaluations"]==row["episode"]*cfg["k"],"validation training-budget provenance")
        require(row["successful_pipeline_evaluations"]+row["failed_pipeline_evaluations"]==row["questions"],"validation outcomes")
    metric="numeric_exact_match_accuracy" if cfg["dataset"]=="gsm8k" else "exact_match_accuracy"
    selected=max(validation,key=lambda row:(row[metric],-row["episode"]))
    best=summary.get("best")
    require(best is not None and best["episode"]==selected["episode"] and best["score"]==selected[metric],"validation-only checkpoint selection")
    tests=[json.loads(path.read_text(encoding="utf-8")) for path in sorted(root.glob("test_*.json"))]
    for result in tests:
        require(result["split"]=="test" and result["questions"]==cfg["test_size"],"held-out test scope")
        require(Path(result["selected_checkpoint"]).resolve()==Path(best["checkpoint"]).resolve(),"held-out checkpoint provenance")
        require(len(result["question_ids"])==len(set(result["question_ids"]))==cfg["test_size"],"held-out question IDs")
        require(not (seen|set(validation_ids)).intersection(result["question_ids"]),"held-out leakage")
        require(result["question_ids"]==tests[0]["question_ids"],"repeated test identity")
    scores=[row[metric] for row in tests]
    accuracy=scores[0] if scores and all(score==scores[0] for score in scores) else None
    require(summary["final_held_out_accuracy"]==accuracy,"held-out summary score")
    if cfg["dataset"]=="hotpotqa":
        require(best.get("f1")==selected["answer_f1"],"selected validation F1")
        scores=[row["answer_f1"] for row in tests]
        f1=scores[0] if scores and all(score==scores[0] for score in scores) else None
        require(summary["final_held_out_f1"]==f1,"held-out summary F1")
    latest=json.loads((root/"latest.json").read_text(encoding="utf-8"))
    require(latest["episode"]==cfg["num_episodes"] and Path(latest["checkpoint"]).is_file(),"latest durable checkpoint")
    require(json.loads((root/"best.json").read_text(encoding="utf-8"))==best and Path(best["checkpoint"]).is_file(),"selected checkpoint pointer")
    metadata_path=root/"config.json"
    require(metadata_path.is_file(),"missing configuration snapshot")
    if metadata_path.exists():
        metadata=json.loads(metadata_path.read_text(encoding="utf-8"))
        require(metadata["config"]==cfg and metadata["selection_id"]==summary["selection_id"],"configuration snapshot")
        require(metadata["initial_policy_digest"]==summary["initial_policy_digest"] and
                metadata["execution_model"]["model"].get("digest")==summary["execution_model_digest"],"model snapshot identity")
        selection=json.loads(Path(metadata["selection_path"]).read_text(encoding="utf-8"))
        require(stable_hash(selection)==summary["selection_id"],"selection manifest checksum")
        for row in train:
            offset=selection["episode_pool_offsets"][row["episode"]-1]
            require(row["question_pool_offset"]==offset and row["question_id"]==selection["ids"]["train"][offset],"persisted episode schedule")
        require(validation_ids==selection["ids"]["validation"],"persisted validation IDs")
        require(all(row["question_ids"]==selection["ids"]["test"] for row in tests),"persisted held-out IDs")
    return dict(train=train,validation=validation)

def compare_runs(run_dirs):
    if not run_dirs:
        raise ValueError("Provide explicit completed run directories")
    runs=[]; reference=None; seen_roots=set()
    for root in map(Path,run_dirs):
        if root.resolve() in seen_roots:
            raise ValueError("Duplicate run directory; provide each run once")
        seen_roots.add(root.resolve())
        summary=json.loads((root/"summary.json").read_text(encoding="utf-8"))
        if not summary["completed"]:
            raise ValueError(f"Run incomplete: {root}")
        cfg={key:value for key,value in summary["config"].items() if key not in ("k","results_root")}
        runtime_identity=summary.get("runtime_identity")
        if runtime_identity is None:
            metadata=json.loads((root/"config.json").read_text(encoding="utf-8"))
            runtime_identity=dict(packages=metadata["hardware"]["packages"],ollama_version=metadata["execution_model"].get("version"))
        identity=dict(config=cfg,selection_id=summary["selection_id"],initial_policy_digest=summary["initial_policy_digest"],
                      execution_model_digest=summary["execution_model_digest"],runtime_identity=runtime_identity)
        if reference is not None and identity!=reference:
            raise ValueError("Runs differ in non-K protocol, data or model identity")
        reference=identity
        history=verify_run_artifacts(root,summary)
        runs.append(dict(summary=summary,**history))
    return sorted(runs,key=lambda run:run["summary"]["config"]["k"])

def run_all_k_sequential(cfg):
    paths=[]; reference_selection=None; reference_initial=None
    for k in ALLOWED_K:
        session=initialize_experiment(dict(cfg,k=k))
        try:
            selection=session["data"]["selection_id"]; initial=session["initial_policy_digest"]
            if reference_selection is not None and (selection!=reference_selection or initial!=reference_initial):
                raise RuntimeError("Independent K initialization/schedule mismatch")
            reference_selection,reference_initial=selection,initial
            train_grpo(session); paths.append(str(session["run_dir"]))
        finally:
            release_experiment(session)
    return paths

# 20. Visualizations
Eleven Matplotlib plots use recorded results only. Missing measurements or tests create an unavailable note without a fabricated point. Figures display here and may be saved. Accuracy versus budget uses cumulative **training** attempts, distinct from evaluation cost. Repeats are displayed separately.

In [ ]:
def plot_comparison(runs,save_dir=None):
    figures=[]
    def finish(fig,name):
        fig.tight_layout(); figures.append(fig)
        if save_dir is not None:
            destination=Path(save_dir); destination.mkdir(parents=True,exist_ok=True)
            fig.savefig(destination/(name+"_"+uuid.uuid4().hex[:8]+".png"),dpi=160)
        plt.show()
    def across_k(title,key,ylabel,best=False):
        fig,ax=plt.subplots(figsize=(6,4)); points=[]
        for run in runs:
            summary=run["summary"]
            value=summary.get("best",{}).get("score") if best and summary.get("best") else summary.get(key) if not best else None
            if value is not None:
                points.append((summary["config"]["k"],value))
        if points:
            ax.scatter(*zip(*points)); ax.set_xticks(ALLOWED_K)
        else:
            ax.text(.5,.5,"No recorded results available",ha="center",transform=ax.transAxes)
        ax.set(title=title,xlabel="K",ylabel=ylabel); finish(fig,key)
    across_k("Validation-selected numeric EM","validation_accuracy","Numeric EM",best=True)
    across_k("Final held-out numeric EM","final_held_out_accuracy","Numeric EM")
    across_k("Training time (excludes validation/checkpoint I/O)","training_time","Seconds")
    across_k("Training candidate attempts","attempted_pipeline_evaluations","Pipelines")
    def curves(title,rows_key,xkey,ykey,ylabel):
        fig,ax=plt.subplots(figsize=(7,4)); present=False
        for index,run in enumerate(runs):
            pairs=[(r[xkey],r[ykey]) for r in run[rows_key] if r.get(xkey) is not None and r.get(ykey) is not None]
            if pairs:
                ax.plot(*zip(*pairs),marker="o",label=f"K={run['summary']['config']['k']} run {index+1}"); present=True
        if present:
            ax.legend()
        else:
            ax.text(.5,.5,"No recorded results available",ha="center",transform=ax.transAxes)
        ax.set(title=title,xlabel=xkey.replace("_"," "),ylabel=ylabel); finish(fig,ykey+"_"+xkey)
    curves("Average reward by episode","train","episode","mean_group_reward","Mean reward")
    across_k("Equal-reward groups","zero_advantage_group_percentage","Percent")
    curves("Gradient norms (unnormalized Figure 2 sum)","train","episode","gradient_norm","L2 gradient norm")
    curves("Validation accuracy by checkpoint","validation","episode","numeric_exact_match_accuracy","Numeric EM")
    curves("Validation accuracy by training budget","validation","cumulative_training_pipeline_evaluations","numeric_exact_match_accuracy","Numeric EM")
    across_k("Total logical LM calls in effective history","total_llm_calls_in_effective_history","LM.forward attempts")
    across_k("Pipeline execution latency","average_pipeline_execution_latency","Seconds per pipeline")
    return figures

# 21. Reproducibility and Experiment Summary
Keep `RUN_OPERATION=False` when running definition cells. Rebuild CONFIG after edits and explicitly enable the operation below when ready. Training does not automatically test. `RUN_ALL_K=True` requests sequential independent training; each policy is released before the next.

Resume/validate/test require SELECTED_CHECKPOINT. For resume use latest.json; for final testing use best.json. Remote inference is not guaranteed deterministic. First-token collisions, last-number extraction, failure rewards and K-dependent gradient scaling remain documented limitations.

Notebook-local synthetic checks below use tiny test policies and mocks, never a substituted research model. They verify mechanics, never generate benchmark results. See [usage guide](STATIC_K_EXPERIMENT_GUIDE.md).

In [ ]:
def run_operation(cfg,mode,checkpoint=None,run_dirs=None,all_k=False):
    if mode=="compare":
        runs=compare_runs(run_dirs or []); plot_comparison(runs); return runs
    if mode not in ("initialize","train","resume","validate","test"):
        raise ValueError("MODE must be initialize/train/resume/validate/test/compare")
    if all_k:
        if mode!="train":
            raise ValueError("RUN_ALL_K requires MODE=train")
        return run_all_k_sequential(cfg)
    if mode in ("resume","validate","test"):
        if checkpoint is None:
            raise ValueError("Set SELECTED_CHECKPOINT to an explicit .pt path")
        session=load_selected_experiment(checkpoint,cfg,for_resume=mode=="resume")
    else:
        session=initialize_experiment(cfg)
    if mode in ("train","resume"):
        train_grpo(session)
    elif mode=="validate":
        validate_selected(session)
    elif mode=="test":
        test_model(session)
    return session

COMPARISON_RUN_DIRS=[]
if RUN_OPERATION:
    CONFIG=build_config()
    if "SESSION" in globals() and isinstance(SESSION,dict) and "policy" in SESSION:
        release_experiment(SESSION)
    SESSION=run_operation(CONFIG,MODE,checkpoint=SELECTED_CHECKPOINT,
                          run_dirs=COMPARISON_RUN_DIRS,all_k=RUN_ALL_K)
else:
    print("Setup complete. No training/test launched. Run notebook_self_checks() for synthetic checks.")

In [ ]:
def phase4_self_checks(session, check):
    from unittest.mock import patch
    root=session["run_dir"]
    original_summary=json.loads((root/"summary.json").read_text(encoding="utf-8"))
    original_train=read_jsonl(root/"training.jsonl")
    original_validation=read_jsonl(root/"validation.jsonl")
    check(len(compare_runs([root]))==1,"comparison verifies persisted manifest and effective history")
    def rejected(operation, name):
        try:
            operation()
        except ValueError:
            check(True,name)
        else:
            raise AssertionError(name)
    rejected(lambda:compare_runs([root,root]),"comparison rejects duplicate run directories")
    for key,value in (("attempted_pipeline_evaluations",1),("optimizer_updates",0),("final_held_out_accuracy",.123)):
        altered=copy.deepcopy(original_summary); altered[key]=value
        rejected(lambda:verify_run_artifacts(root,altered),"comparison rejects summary drift "+key)
    for key,value in (("episode",2),("attempted_pipeline_evaluations",1),("advantages",[1.]*session["config"]["k"]),
                      ("question_id","wrong-question"),("question_pool_offset",999)):
        altered=copy.deepcopy(original_train); altered[0][key]=value
        write_jsonl(root/"training.jsonl",altered)
        try:
            rejected(lambda:verify_run_artifacts(root,original_summary),"comparison rejects episode drift "+key)
        finally:
            write_jsonl(root/"training.jsonl",original_train)
    altered=copy.deepcopy(original_validation); altered[-1]["cumulative_training_pipeline_evaluations"]+=1
    write_jsonl(root/"validation.jsonl",altered)
    try:
        rejected(lambda:verify_run_artifacts(root,original_summary),"comparison rejects accuracy/budget provenance drift")
    finally:
        write_jsonl(root/"validation.jsonl",original_validation)
    for key,value in (("preset","baseline"),("num_episodes",200),("seed",43),("validation_size",2),
                      ("execution_max_tokens",999),("policy_revision","changed")):
        rejected(lambda:compatible_config(session["config"],dict(session["config"],**{key:value})),
                 "checkpoint rejects incompatible "+key)
    check(compare_runs([root])[0]["summary"]==original_summary,"comparison preserves recorded summary")
    metadata=(root/"config.json").read_bytes()
    (root/"config.json").rename(root/"config.missing")
    try:
        rejected(lambda:verify_run_artifacts(root,original_summary),"comparison rejects missing configuration snapshot")
    finally:
        (root/"config.missing").rename(root/"config.json")
    check((root/"config.json").read_bytes()==metadata,"configuration provenance preserved")
    incompatible=root.parent/uuid.uuid4().hex; incompatible.mkdir()
    mismatch=copy.deepcopy(original_summary); mismatch["config"]["seed"]+=1
    write_json(incompatible/"summary.json",mismatch)
    rejected(lambda:compare_runs([root,incompatible]),"comparison rejects cross-run protocol drift")
    totals=recorded_resource_totals(original_train,original_validation,[])
    check(not totals["resource_journal_complete"] and totals["observed_llm_calls_including_rollback"] is None,
          "missing resource journal remains unavailable")
    journal=read_jsonl(root/"attempts.jsonl")
    tests=[json.loads(path.read_text(encoding="utf-8")) for path in root.glob("test_*.json")]
    totals=recorded_resource_totals(original_train,original_validation+tests,journal)
    check(totals["resource_journal_complete"] and totals["observed_pipeline_attempts_including_rollback"]>=
          sum(row["attempted_pipeline_evaluations"] for row in original_train+original_validation+tests),
          "complete journal accounts for observed evaluation and training work")
    check(totals["peak_allocated_gpu_bytes"] is None and totals["observed_prompt_tokens_including_rollback"] is None,
          "unmeasured GPU and tokens remain unavailable")
    check(totals["training_judge_failures"]==sum(len(row["judge_failures"]) for row in original_train),
          "failure aggregation uses recorded events")
    observed=[]; active=[]
    def initialize(cfg):
        if active:
            raise AssertionError("Concurrent policies in all-K execution")
        fresh=dict(config=cfg,data=session["data"],initial_policy_digest=session["initial_policy_digest"],
                   run_dir=root/str(cfg["k"]))
        observed.append(cfg["k"]); active.append(fresh); return fresh
    def release(fresh):
        check(active==[fresh],"all-K releases each initialized policy")
        active.clear()
    with patch.dict(globals(),initialize_experiment=initialize,train_grpo=lambda fresh:fresh,release_experiment=release):
        paths=run_all_k_sequential(session["config"])
    check(observed==list(ALLOWED_K) and len(paths)==4 and not active,"explicit all-K sequencing uses independent starts")
    events=[]
    with patch.dict(globals(),initialize_experiment=lambda cfg:events.append("initialize") or {},
                    load_selected_experiment=lambda path,cfg,for_resume=False:events.append("resume" if for_resume else "selected") or {},
                    train_grpo=lambda fresh:events.append("train"),validate_selected=lambda fresh:events.append("validate"),
                    test_model=lambda fresh:events.append("test"),compare_runs=lambda roots:events.append("compare") or [],
                    plot_comparison=lambda runs:events.append("plot")):
        for mode in ("initialize","train","resume","validate","test","compare"):
            events.clear(); run_operation(session["config"],mode,checkpoint="explicit.pt",run_dirs=[root])
            expected={"initialize":["initialize"],"train":["initialize","train"],"resume":["resume","train"],
                      "validate":["selected","validate"],"test":["selected","test"],"compare":["compare","plot"]}[mode]
            check(events==expected,"explicit operation dispatch "+mode)
        for mode in ("resume","validate","test"):
            rejected(lambda:run_operation(session["config"],mode),"operation requires selected checkpoint "+mode)

def notebook_self_checks():
    from types import SimpleNamespace
    from unittest.mock import patch
    passed=[]
    def check(condition,name):
        if not condition:
            raise AssertionError(name)
        passed.append(name)
    for response,target,expected in [("[1,234]","1234",True),("[-5.50]","-5.5",True),
                                      ("[.5]","0.5",True),("[50]","5",False),
                                      ("[1,999]","1234",False),("reason 2; final 7","7",True),
                                      (None,"0",False),("[nan]","5",False)]:
        check(compare_answers(response,target)==expected,"numeric normalization "+str(response))
    check(canonical_target("work\n#### -1,234.50")=="-1234.5","canonical GSM8K target")
    check(parse_judge_score("[0.1]")==0.1,"fractional judge, no substring false-positive")
    for value in ["[nan]","[inf]","correct 1","[1] [0]","0.1 extra"]:
        try:
            parse_judge_score(value)
        except ValueError:
            passed.append("reject malformed judge "+value)
        else:
            raise AssertionError("Malformed judge accepted")
    class FakeLM:
        def __init__(self): self.events=[]
        def __call__(self,*args,**kwargs):
            self.events.append(dict(seconds=0.0,prompt_tokens=None,generated_tokens=None,error=None))
            return ["[0.5]"]
    check(compute_reward("q","[6]","5",FakeLM())["reward"]==.5,"fractional reward preserved")
    class BrokenLM(FakeLM):
        def __call__(self,*args,**kwargs): raise RuntimeError("service unavailable")
    check(compute_reward("q","[6]","5",BrokenLM())["judge_error"] is not None,"judge failure logged as zero reward")
    for k in ALLOWED_K:
        rewards=[float(i*i)/(k*k) for i in range(k)]
        mean,advantages=group_advantages(rewards)
        check(np.isclose(mean,np.mean(rewards)) and np.isclose(sum(advantages),0),f"K={k} centered rewards")
        theta=torch.tensor(.7,requires_grad=True)
        logps=[[theta*(i+1),theta*(-.25)] for i in range(k)]
        loss=figure2_loss(logps,advantages)
        reference=-sum((i+1-.25)*advantages[i] for i in range(k))
        loss.backward()
        check(np.isclose(float(theta.grad),reference,atol=1e-6) and np.isclose(float(loss.detach()),.7*reference,atol=1e-6),f"K={k} sum loss and gradient, no K division")
        check(np.all(group_advantages([1.]*k)[1]==0),f"K={k} equal rewards")
    fake_data={"train":[dict(question=f"question {i}",answer=f"reason\n#### {i}") for i in range(300)],
               "test":[dict(question=f"test {i}",answer=f"reason\n#### {i}") for i in range(250)]}
    cfg=build_config(dataset_revision="synthetic-only",policy_device="cpu")
    saved_states=rng_state()
    sample_a=(random.random(),float(np.random.random()),float(torch.rand(())))
    restore_rng(saved_states)
    sample_b=(random.random(),float(np.random.random()),float(torch.rand(())))
    check(sample_a==sample_b,"Python NumPy Torch RNG round-trip")
    try:
        build_config(k=6)
    except ValueError:
        passed.append("reject unsupported K")
    else:
        raise AssertionError("Unsupported K accepted")
    baseline,subsets=selection_payload(cfg,fake_data)
    for k in ALLOWED_K:
        other,_=selection_payload(dict(cfg,k=k,preset="pilot",num_episodes=20),fake_data)
        check(other==baseline,f"K={k} identical persisted schedule and subsets")
    check(set(baseline["ids"]["train"]).isdisjoint(baseline["ids"]["validation"]),"train/validation isolation")
    check(len(set(baseline["episode_pool_offsets"]))<len(baseline["episode_pool_offsets"]),"seeded schedule actually repeats")
    check(baseline["episode_pool_offsets"][:5]==selection_payload(dict(cfg,num_episodes=5),fake_data)[0]["episode_pool_offsets"][:5],"smoke prefix")
    class TinyTokenizer:
        def __call__(self,text,**kwargs):
            return dict(input_ids=torch.tensor([[1,2,3]]),attention_mask=torch.ones(1,3,dtype=torch.long))
    class TinyPolicy(torch.nn.Module):
        def __init__(self):
            super().__init__(); self.logits_parameter=torch.nn.Parameter(torch.linspace(-.2,.2,32))
            self.config=SimpleNamespace(n_positions=128)
        def forward(self,input_ids,**kwargs):
            return SimpleNamespace(logits=self.logits_parameter.view(1,1,-1).expand(1,input_ids.shape[1],-1))
    first_tokens={action:i%12 for i,action in enumerate(ACTIONS)}
    def fixture(root):
        config=build_config(k=5,num_episodes=3,train_pool_size=2,validation_size=1,test_size=1,
                            checkpoint_interval=1,validation_interval=1,policy_device="cpu",
                            dataset_revision="synthetic-only",results_root=str(root))
        small={"train":[dict(question=f"synthetic train {i}",answer="work\n#### 0") for i in range(8)],
               "test":[dict(question=f"synthetic test {i}",answer="work\n#### 0") for i in range(4)]}
        data=prepare_selection(config,small)
        root=Path(root)/uuid.uuid4().hex; root.mkdir(); (root/"checkpoints").mkdir()
        model=TinyPolicy().eval()
        write_json(root/"config.json",dict(config=config,selection_id=data["selection_id"],selection_path=data["selection_path"],
                   initial_policy_digest=parameter_digest(model),hardware=hardware_metadata(),execution_model=dict(model=dict(digest="synthetic-only"))))
        return dict(config=config,data=data,policy=model,tokenizer=TinyTokenizer(),first_tokens=first_tokens,
                    optimizer=torch.optim.Adam(model.parameters(),lr=config["learning_rate"]),lm=FakeLM(),
                    llm_metadata=dict(model=dict(digest="synthetic-only")),initial_policy_digest=parameter_digest(model),
                    run_dir=root,in_episode=False,state=dict(episode=0,updates=0,train=[],validation=[],tests=[],best=None,seen=[],cumulative_training_seconds=0.0))
    with tempfile.TemporaryDirectory(prefix="autodspy-synthetic-") as temporary:
        session=fixture(temporary)
        seed_everything(42)
        trajectory=generate_pipeline(session["policy"],session["tokenizer"],first_tokens,"q",session["config"])
        replay=replay_log_probs(session["policy"],session["tokenizer"],first_tokens,"q",trajectory,session["config"])
        check(np.allclose(trajectory["log_probs"],[float(x.detach()) for x in replay]),"rollout and replay log probabilities")
        check(trajectory["actions"][-1]=="stop" and trajectory["log_probs"][-1]==0.0,"deterministic stop")
        for k in ALLOWED_K:
            session["config"]["k"]=k
            seed_everything(42)
            trajectories=[generate_pipeline(session["policy"],session["tokenizer"],first_tokens,"q",session["config"]) for _ in range(k)]
            rewards=[float(i%2) for i in range(k)]
            advantages=group_advantages(rewards)[1]
            session["optimizer"].zero_grad()
            reference=figure2_loss([replay_log_probs(session["policy"],session["tokenizer"],first_tokens,"q",t,session["config"]) for t in trajectories],advantages)
            reference.backward(); expected=[p.grad.clone() for p in session["policy"].parameters()]
            captured=[]
            def capture_step():
                captured.extend(p.grad.clone() for p in session["policy"].parameters())
            with patch.object(session["optimizer"],"step",capture_step):
                result=update_policy(session,"q",trajectories,rewards)
            check(all(torch.allclose(a,b,atol=1e-6) for a,b in zip(expected,captured)),f"K={k} memory-bounded replay equals retained gradient")
            check(result["gradient_norm"]>0,f"K={k} intended policy receives gradient")
        session["config"]["k"]=5
        before=parameter_digest(session["policy"])
        update_policy(session,"q",trajectories[:5],[0.,1.,0.,1.,0.])
        check(before!=parameter_digest(session["policy"]),"Adam updates intended parameters")
        # Fresh rollouts required after a policy step.
        equal=[generate_pipeline(session["policy"],session["tokenizer"],first_tokens,"q",session["config"]) for _ in range(5)]
        equal_result=update_policy(session,"q",equal,[1.]*5)
        check(equal_result["gradient_norm"]==0 and equal_result["optimizer_step"],"equal-reward group preserves finite Adam step")
        fresh=fixture(temporary)
        check(fresh["initial_policy_digest"]==fixture(temporary)["initial_policy_digest"] and not fresh["optimizer"].state,"fresh independent parameters and optimizer")
        # Mock execution only; these transient rows are NOT experimental results.
        def execution(trajectory,prompt,lm):
            lm.events.append(dict(seconds=0.,prompt_tokens=None,generated_tokens=None,error=None))
            return dict(output="[0]" if trajectory["module"]=="CoT" else "[1]",success=True,error=None,seconds=0.)
        policy_ids=[]
        original_generator=generate_pipeline
        def generator(model,*args,**kwargs):
            policy_ids.append(id(model)); return original_generator(model,*args,**kwargs)
        with patch.dict(globals(),execute_pipeline=execution,ollama_snapshot=lambda cfg:{"unavailable":"synthetic test"},generate_pipeline=generator):
            saved=rng_state(); digest=parameter_digest(fresh["policy"])
            result=evaluate_module(fresh)
            check(result["questions"]==1 and result["attempted_pipeline_evaluations"]==1,"one pipeline per validation question")
            check(policy_ids[-1]==id(fresh["policy"]) and digest==parameter_digest(fresh["policy"]),"evaluation uses selected policy without updates")
            check(torch.equal(saved["torch"],rng_state()["torch"]) and saved["python"]==rng_state()["python"],"validation restores training RNG")
            train_grpo(fresh)
            check([r["episode"] for r in fresh["state"]["train"]]==[1,2,3],"one statistical row per episode")
            check(all(r["attempted_pipeline_evaluations"]==5 for r in fresh["state"]["train"]),"exactly K candidates per episode")
            check(len(read_jsonl(fresh["run_dir"]/"training.jsonl"))==3,"structured metric logging")
            best=fresh["state"]["best"]["checkpoint"]
            evaluation=fixture(temporary); load_checkpoint_into(evaluation,best)
            check(parameter_digest(evaluation["policy"])==parameter_digest(torch_load_policy_fixture(best,TinyPolicy)),"selected checkpoint policy restoration")
            for operation in (save_checkpoint,train_grpo):
                try:
                    operation(evaluation)
                except RuntimeError:
                    passed.append("evaluation-only session rejects "+operation.__name__)
                else:
                    raise AssertionError("Evaluation-only state used for training/checkpoint")
            test_model(evaluation)
            complete=compare_runs([fresh["run_dir"]]); check(len(complete)==1,"completed artifact comparison")
            phase4_self_checks(fresh,check)
            full_digest=parameter_digest(fresh["policy"])
            check(full_digest!=fresh["initial_policy_digest"],"synthetic training/resume fixture has nonzero optimizer updates")
            interrupted=fixture(temporary); seed_everything(42)
            # Match the same initial RNG as the uninterrupted run (fresh had evaluation with isolated RNG).
            # Re-run a reference with explicit seed to compare durable recovery exactly.
            reference_session=fixture(temporary); seed_everything(42); train_grpo(reference_session)
            reference_digest=parameter_digest(reference_session["policy"])
            seed_everything(42)
            def interrupt_execution(trajectory,prompt,lm):
                if interrupted["state"]["episode"]==1 and interrupted["in_episode"]:
                    raise KeyboardInterrupt("synthetic interruption during episode 2")
                return execution(trajectory,prompt,lm)
            with patch.dict(globals(),execute_pipeline=interrupt_execution):
                try:
                    train_grpo(interrupted)
                except KeyboardInterrupt:
                    pass
                else:
                    raise AssertionError("Interruption did not occur")
            latest=json.loads((interrupted["run_dir"]/"latest.json").read_text())["checkpoint"]
            load_checkpoint_into(interrupted,latest,for_resume=True)
            check(interrupted["state"]["episode"]==1,"resume starts next unfinished durable episode")
            train_grpo(interrupted)
            check(parameter_digest(interrupted["policy"])==reference_digest,"interrupted resume matches uninterrupted final policy")
            check([r["episode"] for r in interrupted["state"]["train"]]==[1,2,3] and interrupted["state"]["updates"]==3,"resume has no duplicate rows/updates")
            original_evaluator=evaluate_module
            for interrupted_validation_episode in (1,3):
                interrupted_validation=fixture(temporary); seed_everything(42)
                def interrupt_validation(session,*args,**kwargs):
                    if session["state"]["episode"]==interrupted_validation_episode:
                        raise KeyboardInterrupt("synthetic validation interruption")
                    return original_evaluator(session,*args,**kwargs)
                with patch.dict(globals(),evaluate_module=interrupt_validation):
                    try:
                        train_grpo(interrupted_validation)
                    except KeyboardInterrupt:
                        pass
                    else:
                        raise AssertionError("Validation interruption did not occur")
                latest=json.loads((interrupted_validation["run_dir"]/"latest.json").read_text())["checkpoint"]
                load_checkpoint_into(interrupted_validation,latest,for_resume=True)
                train_grpo(interrupted_validation)
                check([row["episode"] for row in interrupted_validation["state"]["validation"]]==[0,1,2,3],f"interrupted validation {interrupted_validation_episode} is retried exactly once")
                check(parameter_digest(interrupted_validation["policy"])==reference_digest and interrupted_validation["state"]["updates"]==3,f"validation interruption {interrupted_validation_episode} causes no duplicate updates")
            try:
                compatible_config(interrupted["config"],dict(interrupted["config"],k=8))
            except ValueError:
                passed.append("reject incompatible K checkpoint")
            else:
                raise AssertionError("K mismatch not rejected")
            test_path=Path(temporary)/"exclusive.pt"; atomic_torch_save(test_path,{"test":1})
            try:
                atomic_torch_save(test_path,{"test":2})
            except FileExistsError:
                passed.append("checkpoint overwrite refused")
            else:
                raise AssertionError("Checkpoint overwritten")
    # Plot mechanics only: no fabricated points or saved synthetic research plots.
    with patch.object(plt,"show",lambda:None):
        figures=plot_comparison([])
    check(len(figures)==11 and all(not figure.axes[0].lines and not figure.axes[0].collections for figure in figures),"eleven empty-data plots, no fake results")
    for figure in figures: plt.close(figure)
    print(f"PASS: {len(passed)} notebook-local synthetic checks. No Llama/benchmark success implied.")
    return passed

def torch_load_policy_fixture(path,policy_class):
    model=policy_class()
    payload=torch.load(path,map_location="cpu",weights_only=False)
    model.load_state_dict(payload["policy"])
    return model

RUN_SELF_CHECKS=False
if RUN_SELF_CHECKS:
    SELF_CHECK_RESULTS=notebook_self_checks()


def full_policy_update_check(cfg):
    """Opt-in engineering check: real pretrained GPT-2, synthetic group, no Llama calls/results."""
    model,tokenizer,optimizer,first_tokens,initial=initialize_policy(cfg)
    session=dict(policy=model,tokenizer=tokenizer,optimizer=optimizer,first_tokens=first_tokens,config=cfg)
    if next(model.parameters()).is_cuda:
        torch.cuda.reset_peak_memory_stats()
    prompt="Synthetic verification only: what is 2 plus 3?"
    trajectories=[]
    try:
        for index in range(cfg["k"]):
            actions=[MODULES[index%2],SIGNATURES[0],"stop"]
            trajectory=dict(actions=actions)
            with torch.no_grad():
                trajectory["log_probs"]=[float(value) for value in replay_log_probs(model,tokenizer,first_tokens,prompt,trajectory,cfg)]
            trajectories.append(trajectory)
        rewards=[1.,0.]+[.5]*(cfg["k"]-2)
        result=update_policy(session,prompt,trajectories,rewards)
        changed=parameter_digest(model)!=initial
        if not changed or result["gradient_norm"]<=0:
            raise AssertionError("Real GPT-2 did not receive nonzero gradients and update")
        return dict(kind="synthetic-group-on-real-GPT2-not-benchmark",passed=True,
                    initial_policy_digest=initial,parameters_changed=changed,**result,**gpu_statistics(model))
    finally:
        release_experiment(session)


# 22. Controlled K=5 stability investigation
The reference run explicitly used Llama 3.2 3B, not the default Llama 3.1 8B. Copy its immutable dataset/schedule, reward, generation settings and CUDA setup. Change only learning rate (1e-5), episode budget (10), and run label. Start pretrained GPT-2 and fresh Adam; retain Figure 2 sum and zero-gradient Adam steps.
Logging probes do not sample or consume RNG. No second pilot, other K, or held-out testing is automatic. Full distributions, pre/post-update entropy and semantic judge traces are saved in the new run. Call `stability_self_checks()` before explicitly running `run_stability_investigation(STABILITY_REFERENCE_RUN)`.


In [ ]:
def stability_self_checks():
    from types import SimpleNamespace
    from unittest.mock import patch
    from litellm import ModelResponse
    checks = []
    class Tokenizer:
        def __call__(self,text,**kwargs):
            return dict(input_ids=torch.zeros((1,3),dtype=torch.long))
    class Policy(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.theta=torch.nn.Parameter(torch.linspace(-.2,.2,len(ACTIONS)))
            self.config=SimpleNamespace(n_positions=64)
        def forward(self,input_ids,**kwargs):
            return SimpleNamespace(logits=self.theta.view(1,1,-1).expand(1,input_ids.shape[1],-1))
    model=Policy().eval(); tokenizer=Tokenizer(); tokens={a:i for i,a in enumerate(ACTIONS)}
    cfg=build_config(k=5,policy_device="cpu")
    session=dict(policy=model,tokenizer=tokenizer,first_tokens=tokens,config=cfg)
    before=rng_state(); digest=parameter_digest(model)
    probe=probe_policy(session,"q")
    assert torch.equal(before["torch"],rng_state()["torch"]) and before["python"]==rng_state()["python"]
    assert np.array_equal(before["numpy"][1],rng_state()["numpy"][1])
    assert parameter_digest(model)==digest
    for distribution in [probe["module"],*probe["signature_given_module"].values()]:
        probs=np.array(list(distribution["probabilities"].values()))
        assert np.isclose(sum(probs),1) and np.isclose(distribution["entropy_nats"],-sum(probs*np.log(probs)))
    checks.append("full-distribution entropy probes preserve parameters and RNG")
    # Replay a reference sampler with the same categorical draws, excluding logging.
    def reference_sample():
        partial=[]; logps=[]
        with torch.no_grad():
            for _ in range(cfg["max_pipeline_steps"]):
                choices,scores=action_log_probs(model,tokenizer,tokens,"q",partial,cfg)
                index=int(torch.multinomial(scores.exp(),1).item())
                partial.append(choices[index]); logps.append(float(scores[index]))
                if partial[-1]=="stop": break
        return partial,logps
    seed_everything(42); expected=reference_sample(); expected_rng=rng_state()
    seed_everything(42); sampled=sample_actions(model,tokenizer,tokens,"q",cfg)
    assert (sampled["actions"],sampled["log_probs"])==expected
    assert torch.equal(expected_rng["torch"],rng_state()["torch"])
    assert all(np.isclose(d["selected_probability"],math.exp(lp)) for d,lp in zip(sampled["action_diagnostics"],sampled["log_probs"]))
    assert sampled["action_diagnostics"][-1]["entropy_nats"]==0
    checks.append("rollout diagnostics preserve actions, log probabilities and RNG draws")
    lm=MeasuredLM("ollama_chat/synthetic-only",api_key="synthetic-secret",seed=42,
                  temperature=0.,max_tokens=64,num_ctx=4096,cache=False,num_retries=0)
    def reply(text):
        return ModelResponse(model="synthetic-only",choices=[dict(index=0,finish_reason="stop",
               message=dict(role="assistant",content=text))],usage=dict(prompt_tokens=3,completion_tokens=2,total_tokens=5))
    kwargs_before=copy.deepcopy(lm.kwargs)
    with patch.object(dspy.LM,"forward",return_value=reply("[0.5]")):
        reward=compute_reward("q","[6]","5",lm)
    trace=reward["semantic_judge"]
    assert reward["reward"]==.5 and trace["raw_responses"]==["[0.5]"]
    assert trace["prompt"].endswith("final score:[]") and "Ground Truth: 5" in trace["prompt"]
    assert trace["provider_calls"][0]["judge_provider_response"]["choices"][0]["message"]["content"]=="[0.5]"
    assert trace["provider_calls"][0]["judge_request"]["inference"]["parameters"]["seed"]==42
    assert "synthetic-secret" not in json.dumps(trace)
    assert lm.kwargs==kwargs_before and not lm.capture_semantic_judge
    checks.append("judge raw prompt/outputs/provider response/settings saved without secrets or request changes")
    with patch.object(dspy.LM,"forward",return_value=reply("invalid score")):
        bad=compute_reward("q","[6]","5",lm)
    assert bad["reward"]==0 and bad["judge_error"] and bad["semantic_judge"]["raw_responses"]==["invalid score"]
    with patch.object(dspy.LM,"forward",side_effect=RuntimeError("synthetic service error")):
        bad=compute_reward("q","[6]","5",lm)
    assert bad["semantic_judge"]["provider_calls"][0]["error"] and not lm.capture_semantic_judge
    with patch.object(dspy.LM,"forward",side_effect=AssertionError("No judge for numeric EM")):
        assert compute_reward("q","[5]","5",lm)["reward"]==1
    checks.append("judge parse/transport failures retain traces; numeric EM still bypasses judge")
    print("PASS:",len(checks),"stability instrumentation checks (mocked provider only)",flush=True)
    return checks


def file_sha256(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk:=stream.read(4*1024*1024): digest.update(chunk)
    return digest.hexdigest()


def stability_reference_hashes(reference):
    roots=[Path(reference),Path("docs/diagnostics/k5-20261013")]
    paths=[p for root in roots for p in root.rglob("*") if p.is_file()]
    meta=json.loads((Path(reference)/"config.json").read_text(encoding="utf-8"))
    paths.extend([Path(meta["selection_path"]),Path("DSPy_GRPO.ipynb")])
    return {str(p):file_sha256(p) for p in paths}


def compare_stability_runs(reference, pilot):
    reference,pilot=Path(reference),Path(pilot)
    old_meta=json.loads((reference/"config.json").read_text(encoding="utf-8"))
    new_meta=json.loads((pilot/"config.json").read_text(encoding="utf-8"))
    allowed={"learning_rate","num_episodes","preset"}
    differences={k:dict(reference=old_meta["config"].get(k),pilot=new_meta["config"].get(k))
                 for k in old_meta["config"] if old_meta["config"].get(k)!=new_meta["config"].get(k)}
    if set(differences)-allowed or old_meta["selection_id"]!=new_meta["selection_id"]:
        raise RuntimeError("Stability comparison contains uncontrolled config/schedule changes")
    if old_meta["initial_policy_digest"]!=new_meta["initial_policy_digest"]:
        raise RuntimeError("Stability pilot did not use the original pretrained policy")
    old=read_jsonl(reference/"training.jsonl"); new=read_jsonl(pilot/"training.jsonl")
    if len(new)!=10 or [r["episode"] for r in new]!=list(range(1,11)):
        raise RuntimeError("Compare only a completed 10-episode stability pilot")
    records=[]
    for a,b in zip(old[:10],new):
        if a["question_id"]!=b["question_id"]: raise RuntimeError("Training question schedule differs")
        unique=lambda r:len({tuple(x) for x in r["actions"]})
        records.append(dict(episode=b["episode"],reference_rewards=a["rewards"],pilot_rewards=b["rewards"],
            reference_unique=unique(a),pilot_unique=unique(b),
            reference_gradient=f"{a['gradient_norm']:.9e}",pilot_gradient=f"{b['gradient_norm']:.9e}",
            reference_loss=f"{a['grpo_loss']:.9e}",pilot_loss=f"{b['grpo_loss']:.9e}",
            reference_zero_variance=a["reward_variance"]==0,pilot_zero_variance=b["reward_variance"]==0,
            pilot_module_entropy_before=b["module_entropy_nats"],
            pilot_module_entropy_after=b["policy_diagnostics"]["after_update"]["module"]["entropy_nats"],
            pilot_signature_entropy_before=b["signature_entropy_nats"],
            pilot_signature_entropy_after=b["policy_diagnostics"]["after_update"]["expected_signature_entropy_nats"]))
    # This explicitly stated cutoff separates 1e-11 cancellation residuals from informative gradients.
    threshold=1e-6
    tail=[]
    for r in reversed(new):
        if len({tuple(x) for x in r["actions"]})!=1: break
        tail.append(r["episode"])
    persistent_singleton_tail=len(tail)>=3
    judge_groups={}
    for r in new:
        for c in r["candidate_details"]:
            if c["reward"]["semantic_used"]:
                trace=c["reward"]["semantic_judge"]
                key=(r["question_id"],trace["prompt"])
                judge_groups.setdefault(key,[]).append(dict(episode=r["episode"],reward=c["reward"]["reward"],
                                                           raw_responses=trace["raw_responses"]))
    conflicts=[dict(question_id=k[0],judge_prompt=k[1],observations=v) for k,v in judge_groups.items()
               if len({item["reward"] for item in v})>1]
    stats=dict(reference=str(reference),pilot=str(pilot),config_differences=differences,
        compared_episodes=10,meaningful_gradient_threshold=threshold,
        reference_diverse_episodes_after_first=sum(r["reference_unique"]>1 for r in records[1:]),
        pilot_diverse_episodes_after_first=sum(r["pilot_unique"]>1 for r in records[1:]),
        reference_meaningful_gradient_episodes=sum(r["gradient_norm"]>threshold for r in old[:10]),
        pilot_meaningful_gradient_episodes=sum(r["gradient_norm"]>threshold for r in new),
        pilot_meaningful_gradient_episodes_after_first=sum(r["gradient_norm"]>threshold for r in new[1:]),
        persistent_singleton_tail=persistent_singleton_tail,singleton_tail_episodes=sorted(tail),
        semantic_judge_conflicts=conflicts,
        recommendation=("Recommend a separate fresh 1e-6 pilot; do not run automatically." if persistent_singleton_tail
                        else "No persistent singleton tail observed; retain this short pilot as limited stability evidence."),
        limitations=["One seed and ten episodes cannot establish baseline performance or optimal learning rate.",
                     "Semantic judging/execution are not fully reproducible despite fixed seed and temperature zero.",
                     "Historical full entropy logs are unavailable; same-question initial/post-first-update evidence is in the preserved diagnosis."],
        episodes=records)
    write_json(pilot/"stability-comparison.json",stats)
    import csv
    with (pilot/"stability-comparison.csv").open("w",newline="",encoding="utf-8") as stream:
        writer=csv.DictWriter(stream,fieldnames=list(records[0]));writer.writeheader();writer.writerows(records)
    print("STABILITY:",json.dumps({k:v for k,v in stats.items() if k not in ("episodes","semantic_judge_conflicts")}),flush=True)
    return stats


def run_stability_investigation(reference, learning_rate=1e-5, episodes=10):
    # Explicitly bounded investigation; never invokes all-K, full test, or another pilot.
    if episodes!=10 or learning_rate!=1e-5:
        raise ValueError("This authorized investigation is exactly 10 episodes at 1e-5")
    reference=Path(reference)
    original=json.loads((reference/"config.json").read_text(encoding="utf-8"))
    if original["config"]["k"]!=5 or original["config"]["dataset"]!="gsm8k":
        raise ValueError("Expected the diagnosed K=5 GSM8K run")
    protected=stability_reference_hashes(reference)
    cfg=copy.deepcopy(original["config"])
    cfg.update(learning_rate=learning_rate,num_episodes=episodes,preset="pilot")
    session=None
    try:
        session=initialize_experiment(cfg)
        if session["initial_policy_digest"]!=original["initial_policy_digest"] or session["optimizer"].state:
            raise RuntimeError("Must start from original pretrained GPT-2 and empty Adam state")
        saved_selection=json.loads(Path(original["selection_path"]).read_text(encoding="utf-8"))
        if session["data"]["selection"]!=saved_selection or session["data"]["selection_id"]!=original["selection_id"]:
            raise RuntimeError("Saved training selection/schedule mismatch")
        if (session["llm_metadata"]["model"].get("digest")!=original["execution_model"]["model"].get("digest") or
            session["llm_metadata"]["version"]!=original["execution_model"]["version"] or
            VERSIONS!=original["hardware"]["packages"]):
            raise RuntimeError("Execution-model/dependency fingerprint differs; no inference permitted")
        write_json(session["run_dir"]/"stability-investigation.json",dict(reference=str(reference),
            changed_controls={key:dict(reference=original["config"][key],pilot=cfg[key])
                              for key in ("learning_rate","num_episodes","preset")},
            original_pretrained_digest_verified=True,fresh_adam_state_verified=True,
            saved_schedule_verified=True,logging_version="stability-observations-v1",protected_hashes=protected,
            training_pipeline_budget=50,validation_pipeline_budget=20,test_pipeline_budget=0,
            methodology="Figure 2 sum and finite Adam steps unchanged; logging is observational"))
        print("PILOT_DIRECTORY:",session["run_dir"],flush=True)
        train_grpo(session)
        return session["run_dir"],compare_stability_runs(reference,session["run_dir"])
    finally:
        if session is not None: release_experiment(session)
        if stability_reference_hashes(reference)!=protected:
            raise RuntimeError("Historical run/diagnosis/reference notebook changed during investigation")


STABILITY_REFERENCE_RUN=Path("results/gsm8k/k5/seed42/baseline/20261013T002408-4024eb86")
RUN_STABILITY_INVESTIGATION=False  # explicit bounded run only; leave disabled in saved notebook
if RUN_STABILITY_INVESTIGATION:
    if "SESSION" in globals() and isinstance(SESSION,dict) and "policy" in SESSION:
        release_experiment(SESSION)
    STABILITY_RUN_DIR,STABILITY_COMPARISON=run_stability_investigation(STABILITY_REFERENCE_RUN)


# 23. Completed stability investigation
Pilot artifact directory: `results/gsm8k/k5/seed42/pilot/20261013T032749-85f3db7c`.
Unique pipelines by episode: **5,4,4,4,4,3,4,5,3,4**. Diversity survived all nine episodes after episode 1. Six episodes had gradients >1e-6, versus one in the matched original prefix; the remaining four had equal rewards. No persistent collapse was observed in this 10-episode window.
See [full report](docs/STABILITY_PILOT_REPORT.md) for controls, verification and limits. This is stability evidence, not a full-baseline accuracy claim. The original run, diagnostics, Figure 2 loss and finite Adam-step behavior remain unchanged. No 1e-6 pilot or full training was launched.
